In [9]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

WILD_ROOT = (
    PROJECT_ROOT
    / "WildDeepfake"
)

WILD_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:")
print(PROJECT_ROOT)

print("\nWildDeepfake root:")
print(WILD_ROOT)

Project root:
/home/jovyan/Deepfake_Low Resource_Low Resolution

WildDeepfake root:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake


In [10]:
import subprocess

command = [
    "hf",
    "download",
    "xingjunm/WildDeepfake",
    "--repo-type",
    "dataset",
    "--include",
    "deepfake_in_the_wild/real_test/*",
    "--local-dir",
    str(WILD_ROOT),
]

print("Downloading WildDeepfake real_test...")

subprocess.run(
    command,
    check=True,
)

print("\nREAL TEST DOWNLOAD COMPLETE")

Fetching 42 files:  36%|███▌      | 15/42 [00:00<00:00, 148.75it/s]

✓ Downloaded
  path: /home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake

REAL TEST DOWNLOAD COMPLETE


Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 179.22it/s]


In [11]:
import subprocess

command = [
    "hf",
    "download",
    "xingjunm/WildDeepfake",
    "--repo-type",
    "dataset",
    "--include",
    "deepfake_in_the_wild/fake_test/*",
    "--local-dir",
    str(WILD_ROOT),
]

print("Downloading WildDeepfake fake_test...")

subprocess.run(
    command,
    check=True,
)

print("\nFAKE TEST DOWNLOAD COMPLETE")

✓ Downloaded
  path: /home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake

FAKE TEST DOWNLOAD COMPLETE


Fetching 115 files: 100%|██████████| 115/115 [00:00<00:00, 1012.29it/s]


In [12]:
from pathlib import Path

dataset_root = (
    WILD_ROOT
    / "deepfake_in_the_wild"
)

real_archives = sorted(
    (
        dataset_root
        / "real_test"
    ).glob("*.tar.gz")
)

fake_archives = sorted(
    (
        dataset_root
        / "fake_test"
    ).glob("*.tar.gz")
)

print("Real test archives:")
print(len(real_archives))

print("\nFake test archives:")
print(len(fake_archives))

print("\nFirst few real archives:")
for path in real_archives[:5]:
    print(path)

print("\nFirst few fake archives:")
for path in fake_archives[:5]:
    print(path)

Real test archives:
42

Fake test archives:
115

First few real archives:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test/104.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test/12.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test/13.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test/15.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test/16.tar.gz

First few fake archives:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_test/1.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_test/10.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_test/100.tar.gz
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deep

In [24]:
from pathlib import Path
import shutil
import tarfile


PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

dataset_root = (
    PROJECT_ROOT
    / "WildDeepfake"
    / "deepfake_in_the_wild"
)

real_archive_dir = (
    dataset_root
    / "real_test"
)

real_extract_dir = (
    dataset_root
    / "extracted"
    / "real_test"
)

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp",
}


# ============================================================
# Find all real_test archives
# ============================================================

def archive_number(path):
    try:
        return int(
            path.name.removesuffix(".tar.gz")
        )
    except ValueError:
        return 10**12


real_archives = sorted(
    real_archive_dir.glob("*.tar.gz"),
    key=archive_number,
)

if not real_archives:
    raise RuntimeError(
        "No real_test archives were found."
    )

print("=" * 80)
print("FULL WILDDEEPFAKE REAL_TEST EXTRACTION")
print("=" * 80)

print("\nArchives found:")
print(len(real_archives))


# ============================================================
# Remove previous extraction completely
# ============================================================

if real_extract_dir.exists():
    print("\nRemoving previous extracted real_test folder:")
    print(real_extract_dir)

    shutil.rmtree(
        real_extract_dir
    )

real_extract_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nFresh extraction directory created.")


# ============================================================
# Extract every archive
# ============================================================

failed_archives = []

for index, archive_path in enumerate(
    real_archives,
    start=1,
):
    number = archive_number(
        archive_path
    )

    destination = (
        real_extract_dir
        / str(number)
    )

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        # r:* automatically detects plain tar,
        # gzip tar, bz2 tar, xz tar, etc.
        with tarfile.open(
            archive_path,
            mode="r:*",
        ) as archive:

            archive.extractall(
                destination
            )

    except Exception as error:
        failed_archives.append(
            {
                "archive": number,
                "path": str(archive_path),
                "error": repr(error),
            }
        )

        if destination.exists():
            shutil.rmtree(
                destination
            )

        print(
            f"\nFAILED: archive {number}"
        )
        print(repr(error))

    print(
        f"\rExtracting real_test: "
        f"{index}/{len(real_archives)} "
        f"| archive {number}",
        end="",
        flush=True,
    )


print("\n")


# ============================================================
# Count extracted images
# ============================================================

real_images = [
    path
    for path in real_extract_dir.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in image_extensions
    )
]

print("=" * 80)
print("EXTRACTION SUMMARY")
print("=" * 80)

print("\nTotal archives:")
print(len(real_archives))

print("\nFailed archives:")
print(len(failed_archives))

print("\nExtracted real images:")
print(f"{len(real_images):,}")


# ============================================================
# Show failures, if any
# ============================================================

if failed_archives:
    print("\nArchives that failed:")

    for item in failed_archives:
        print(
            f"\nArchive {item['archive']}"
        )
        print(item["path"])
        print(item["error"])


# ============================================================
# Show example extracted paths
# ============================================================

print("\nExample extracted images:")

for path in real_images[:10]:
    print(path)


# ============================================================
# Final assessment
# ============================================================

print("\n" + "=" * 80)

if failed_archives:
    print("REAL_TEST EXTRACTION FINISHED WITH ERRORS")
    print("=" * 80)

    print(
        "\nSome archives failed. The list above shows "
        "which ones need attention."
    )

else:
    print("REAL_TEST EXTRACTION COMPLETE")
    print("=" * 80)

    print(
        "\nPASS: every downloaded real_test archive "
        "was extracted."
    )

FULL WILDDEEPFAKE REAL_TEST EXTRACTION

Archives found:
42

Removing previous extracted real_test folder:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test

Fresh extraction directory created.
Extracting real_test: 42/42 | archive 104

EXTRACTION SUMMARY

Total archives:
42

Failed archives:
0

Extracted real images:
58,659

Example extracted images:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test/3/3/real/110/2422.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test/3/3/real/110/2395.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test/3/3/real/110/2351.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test/3/3/real/110/2332.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extra

In [23]:
from pathlib import Path
import tarfile

import pandas as pd


PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

dataset_root = (
    PROJECT_ROOT
    / "WildDeepfake"
    / "deepfake_in_the_wild"
)

real_archive_dir = (
    dataset_root
    / "real_test"
)

real_extract_dir = (
    dataset_root
    / "extracted"
    / "real_test"
)

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp",
}


# ============================================================
# Find all downloaded real archives
# ============================================================

def archive_number(path):
    try:
        return int(
            path.name.removesuffix(".tar.gz")
        )
    except ValueError:
        return 10**12


real_archives = sorted(
    real_archive_dir.glob("*.tar.gz"),
    key=archive_number,
)

if not real_archives:
    raise RuntimeError(
        "No WildDeepfake real_test archives were found."
    )

print("=" * 80)
print("AUDITING WILDDEEPFAKE REAL_TEST EXTRACTION")
print("=" * 80)

print("\nArchives found:")
print(len(real_archives))


# ============================================================
# Audit each archive
# ============================================================

audit_rows = []

for index, archive_path in enumerate(
    real_archives,
    start=1,
):
    number = archive_number(
        archive_path
    )

    destination = (
        real_extract_dir
        / str(number)
    )

    expected_images = None
    archive_error = None

    # --------------------------------------------------------
    # Count images that SHOULD exist according to the archive
    # --------------------------------------------------------

    try:
        with tarfile.open(
            archive_path,
            mode="r:*",
        ) as archive:

            expected_images = sum(
                1
                for member in archive
                if (
                    member.isfile()
                    and Path(
                        member.name
                    ).suffix.lower()
                    in image_extensions
                )
            )

    except Exception as error:
        archive_error = repr(error)

    # --------------------------------------------------------
    # Count images actually extracted
    # --------------------------------------------------------

    if destination.exists():
        extracted_images = sum(
            1
            for path in destination.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in image_extensions
            )
        )
    else:
        extracted_images = 0

    # --------------------------------------------------------
    # Determine status
    # --------------------------------------------------------

    if archive_error is not None:
        status = "ARCHIVE_ERROR"

    elif not destination.exists():
        status = "NOT_EXTRACTED"

    elif expected_images == 0:
        status = "NO_IMAGES_IN_ARCHIVE"

    elif extracted_images == expected_images:
        status = "OK"

    elif extracted_images == 0:
        status = "EMPTY"

    elif extracted_images < expected_images:
        status = "INCOMPLETE"

    else:
        status = "EXTRA_FILES"

    audit_rows.append(
        {
            "archive": number,
            "archive_file": archive_path.name,
            "archive_size_mb": round(
                archive_path.stat().st_size
                / (1024 ** 2),
                2,
            ),
            "expected_images": expected_images,
            "extracted_images": extracted_images,
            "difference": (
                None
                if expected_images is None
                else extracted_images
                - expected_images
            ),
            "status": status,
            "archive_error": archive_error,
        }
    )

    print(
        f"\rChecked "
        f"{index}/{len(real_archives)} "
        f"| archive {number} "
        f"| {status:14s}",
        end="",
        flush=True,
    )


print("\n")


# ============================================================
# Results
# ============================================================

audit = pd.DataFrame(
    audit_rows
)

print("=" * 80)
print("REAL_TEST EXTRACTION AUDIT RESULTS")
print("=" * 80)

status_summary = (
    audit["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="archives")
)

print("\nArchive status summary:")
display(status_summary)


# ============================================================
# Show problems only
# ============================================================

problems = audit[
    audit["status"] != "OK"
].copy()

if len(problems) == 0:
    print(
        "\nPASS: every real_test archive appears "
        "to be completely extracted."
    )

else:
    print(
        f"\nWARNING: {len(problems)} archive(s) "
        "need attention."
    )

    display(
        problems[
            [
                "archive",
                "expected_images",
                "extracted_images",
                "difference",
                "status",
                "archive_error",
            ]
        ]
    )


# ============================================================
# Overall image counts
# ============================================================

valid_expected = pd.to_numeric(
    audit["expected_images"],
    errors="coerce",
)

total_expected = int(
    valid_expected.fillna(0).sum()
)

total_extracted = int(
    audit["extracted_images"].sum()
)

print("\nExpected real images across all archives:")
print(f"{total_expected:,}")

print("\nActually extracted real images:")
print(f"{total_extracted:,}")

print("\nDifference:")
print(
    f"{total_extracted - total_expected:+,}"
)


# ============================================================
# Save audit report
# ============================================================

audit_path = (
    dataset_root
    / "real_test_extraction_audit.csv"
)

audit.to_csv(
    audit_path,
    index=False,
)

print("\nAudit saved to:")
print(audit_path)


# ============================================================
# Final assessment
# ============================================================

print("\n" + "=" * 80)

if len(problems) == 0:
    print("WILDDEEPFAKE REAL_TEST EXTRACTION VERIFIED")
    print("=" * 80)

    print(
        "\nPASS: all downloaded real_test archives "
        "match their extracted image counts."
    )

else:
    print("WILDDEEPFAKE REAL_TEST EXTRACTION INCOMPLETE")
    print("=" * 80)

    print(
        "\nDo not build the WildDeepfake manifest yet."
    )

    print(
        "The table above identifies exactly which "
        "archive numbers need to be repaired."
    )

AUDITING WILDDEEPFAKE REAL_TEST EXTRACTION

Archives found:
42
Checked 42/42 | archive 104 | EXTRA_FILES   

REAL_TEST EXTRACTION AUDIT RESULTS

Archive status summary:


,status,archives
0,NOT_EXTRACTED,41
1,EXTRA_FILES,1


,archive,expected_images,extracted_images,difference,status,archive_error
0,3,414,0,-414,NOT_EXTRACTED,None
1,6,910,0,-910,NOT_EXTRACTED,None
2,8,109,0,-109,NOT_EXTRACTED,None
3,9,76,0,-76,NOT_EXTRACTED,None
4,12,1245,0,-1245,NOT_EXTRACTED,None
5,13,3655,0,-3655,NOT_EXTRACTED,None
6,15,1905,0,-1905,NOT_EXTRACTED,None
7,16,806,0,-806,NOT_EXTRACTED,None
8,17,654,0,-654,NOT_EXTRACTED,None
9,19,1004,0,-1004,NOT_EXTRACTED,None



Expected real images across all archives:
58,659

Actually extracted real images:
1,693

Difference:
-56,966

Audit saved to:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test_extraction_audit.csv

WILDDEEPFAKE REAL_TEST EXTRACTION INCOMPLETE

Do not build the WildDeepfake manifest yet.
The table above identifies exactly which archive numbers need to be repaired.


In [22]:
!pip install pandas

  Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)
Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)


In [19]:
from pathlib import Path
import shutil
import tarfile


PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

dataset_root = (
    PROJECT_ROOT
    / "WildDeepfake"
    / "deepfake_in_the_wild"
)

fake_archive_dir = (
    dataset_root
    / "fake_test"
)

fake_extract_dir = (
    dataset_root
    / "extracted"
    / "fake_test"
)

fake_extract_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Locate fake_test archives
# ============================================================

fake_archives = list(
    fake_archive_dir.glob("*.tar.gz")
)


def archive_number(path):
    try:
        return int(
            path.name.removesuffix(".tar.gz")
        )
    except ValueError:
        return 10**12


fake_archives = sorted(
    fake_archives,
    key=archive_number,
)

if not fake_archives:
    raise RuntimeError(
        "No fake_test .tar.gz archives were found."
    )

print("=" * 80)
print("WILDDEEPFAKE FAKE_TEST EXTRACTION")
print("=" * 80)

print("\nFake archives found:")
print(len(fake_archives))


# ============================================================
# Extract using automatic format detection
# ============================================================

for index, archive_path in enumerate(
    fake_archives,
    start=1,
):
    number = archive_number(
        archive_path
    )

    destination = (
        fake_extract_dir
        / str(number)
    )

    # Skip an archive if this destination already contains files.
    if (
        destination.exists()
        and any(destination.rglob("*"))
    ):
        print(
            f"\rFake extraction: "
            f"{index}/{len(fake_archives)} "
            f"| archive {number} | already extracted",
            end="",
            flush=True,
        )
        continue

    # Remove empty or partial destination.
    if destination.exists():
        shutil.rmtree(
            destination
        )

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        # r:* automatically handles both true .tar.gz files
        # and uncompressed TAR files with misleading .tar.gz names.
        with tarfile.open(
            archive_path,
            mode="r:*",
        ) as archive:

            archive.extractall(
                destination
            )

    except Exception as error:
        if destination.exists():
            shutil.rmtree(
                destination
            )

        print(
            f"\n\nFAILED on fake archive {number}:"
        )
        print(archive_path)
        print(repr(error))

        raise

    print(
        f"\rFake extraction: "
        f"{index}/{len(fake_archives)} "
        f"| archive {number}",
        end="",
        flush=True,
    )


print("\n\n" + "=" * 80)
print("FAKE_TEST EXTRACTION COMPLETE")
print("=" * 80)


# ============================================================
# Verify extracted fake images
# ============================================================

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp",
}

fake_images = [
    path
    for path in fake_extract_dir.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in image_extensions
    )
]

print("\nExtracted fake images:")
print(f"{len(fake_images):,}")

print("\nExample paths:")

for path in fake_images[:10]:
    print(path)

if len(fake_images) == 0:
    raise RuntimeError(
        "Extraction completed but no fake images were found."
    )

print(
    "\nPASS: WildDeepfake fake_test extraction completed."
)

WILDDEEPFAKE FAKE_TEST EXTRACTION

Fake archives found:
115
Fake extraction: 115/115 | archive 115

FAKE_TEST EXTRACTION COMPLETE

Extracted fake images:
107,003

Example paths:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1122.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1148.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1057.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1064.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1168.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test/1/1/fake/33/1159.png
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_

In [31]:
# ============================================================
# DF40 -> WildDeepfake zero-shot cross-dataset evaluation
#
# Uses:
#   - DF40-trained best checkpoints
#   - WildDeepfake real_test + fake_test only
#   - DF40 validation-derived threshold / score direction
#
# NO TRAINING OR THRESHOLD TUNING IS PERFORMED ON WILDDEEPFAKE
# ============================================================

from pathlib import Path
from types import SimpleNamespace
from datetime import datetime

import argparse
import gc
import hashlib
import importlib.util
import json
import math
import os
import time
import traceback

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
)


# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

TRAINING_SCRIPT = (
    PROJECT_ROOT
    / "agrfn_adaptive_routing_df40.py"
)

ABLATION_ROOT = (
    PROJECT_ROOT
    / "outputs_agrfn_ablation_suite"
)

WILD_ROOT = (
    PROJECT_ROOT
    / "WildDeepfake"
    / "deepfake_in_the_wild"
    / "extracted"
)

REAL_TEST_ROOT = (
    WILD_ROOT
    / "real_test"
)

FAKE_TEST_ROOT = (
    WILD_ROOT
    / "fake_test"
)

WILD_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs_wilddeepfake"
)

WILD_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    WILD_OUTPUT_ROOT
    / "wilddeepfake_test_manifest.csv"
)

COMBINED_SUMMARY_PATH = (
    WILD_OUTPUT_ROOT
    / "wilddeepfake_cross_dataset_summary.csv"
)

STATUS_PATH = (
    WILD_OUTPUT_ROOT
    / "wilddeepfake_evaluation_status.csv"
)


# ------------------------------------------------------------
# Set to None to automatically evaluate every E* experiment
# that contains best_checkpoint.pt.
#
# Or specify models manually, e.g.:
#
# EXPERIMENTS = [
#     "E0_late_fusion",
#     "E6_full_agrfn",
# ]
# ------------------------------------------------------------

EXPERIMENTS = None


# ------------------------------------------------------------
# Full paper evaluation:
#
# Keep both at 0.
#
# For a quick smoke test you could temporarily use:
#
# FRAMES_PER_UNIT = 2
# LIMIT_UNITS = 100
#
# Then change them back to 0 for final results.
# ------------------------------------------------------------

FRAMES_PER_UNIT = 0
LIMIT_UNITS = 0


# ------------------------------------------------------------
# None = use checkpoint settings
# ------------------------------------------------------------

BATCH_SIZE = None
NUM_WORKERS = None


# ------------------------------------------------------------
# Threshold used only to summarize model uncertainty.
# It does NOT affect fake/real classification.
# ------------------------------------------------------------

UNCERTAINTY_THRESHOLD = 0.50


# ------------------------------------------------------------
# Important:
#
# For a proper zero-shot test we do NOT silently fall back to
# 0.5 if the model's DF40 decision threshold cannot be found.
# ------------------------------------------------------------

ALLOW_THRESHOLD_05_FALLBACK = False


# ------------------------------------------------------------
# If True, continue testing other models if one fails.
# ------------------------------------------------------------

CONTINUE_ON_MODEL_ERROR = True


# ============================================================
# Initial checks
# ============================================================

required_paths = [
    TRAINING_SCRIPT,
    ABLATION_ROOT,
    REAL_TEST_ROOT,
    FAKE_TEST_ROOT,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 90)
print("DF40 -> WILDDEEPFAKE ZERO-SHOT CROSS-DATASET EVALUATION")
print("=" * 90)

print("\nDevice:")
print(device)

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

print("\nReal test root:")
print(REAL_TEST_ROOT)

print("\nFake test root:")
print(FAKE_TEST_ROOT)

print("\nNo WildDeepfake training or threshold tuning will be performed.")


# ============================================================
# Import A-GRFN module
# ============================================================

module_name = (
    "agrfn_wilddeepfake_eval_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

spec = importlib.util.spec_from_file_location(
    module_name,
    TRAINING_SCRIPT,
)

if spec is None or spec.loader is None:
    raise RuntimeError(
        "Could not import agrfn_adaptive_routing_df40.py"
    )

agrfn = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(
    agrfn
)

print("\nPASS: A-GRFN module imported.")


# ============================================================
# Image discovery
# ============================================================

IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp",
}


def discover_images(root):
    return sorted(
        [
            path
            for path in root.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        ]
    )


real_images = discover_images(
    REAL_TEST_ROOT
)

fake_images = discover_images(
    FAKE_TEST_ROOT
)

if len(real_images) == 0:
    raise RuntimeError(
        "No WildDeepfake real_test images were found."
    )

if len(fake_images) == 0:
    raise RuntimeError(
        "No WildDeepfake fake_test images were found."
    )

print("\nWildDeepfake images discovered:")
print(f"  Real: {len(real_images):,}")
print(f"  Fake: {len(fake_images):,}")
print(f"  Total: {len(real_images) + len(fake_images):,}")


# ============================================================
# Build sequence/unit identifiers
#
# WildDeepfake images are organized in face-sequence folders.
# The immediate parent directory therefore acts as the
# sequence identifier.
#
# Prefixing with the class avoids accidental collisions
# between real and fake sequence names.
# ============================================================

def sequence_identifier(
    image_path,
    class_root,
    label_name,
):
    relative_parent = (
        image_path.parent
        .relative_to(class_root)
    )

    return (
        f"{label_name}::"
        f"{relative_parent.as_posix()}"
    )


manifest_rows = []


for path in real_images:

    unit_id = sequence_identifier(
        path,
        REAL_TEST_ROOT,
        "real",
    )

    manifest_rows.append(
        {
            "image_path": str(
                path.resolve()
            ),
            "label": 0,
            "label_name": "real",
            "split": "external_test",
            "sample_unit_id": unit_id,
            "group_id": unit_id,
            "video_id": unit_id,
            "method": "wilddeepfake",
            "domain": "wilddeepfake",
            "dataset_partition": (
                "wilddeepfake_real_test"
            ),
            "top_level_folder": (
                "real_test"
            ),
        }
    )


for path in fake_images:

    unit_id = sequence_identifier(
        path,
        FAKE_TEST_ROOT,
        "fake",
    )

    manifest_rows.append(
        {
            "image_path": str(
                path.resolve()
            ),
            "label": 1,
            "label_name": "fake",
            "split": "external_test",
            "sample_unit_id": unit_id,
            "group_id": unit_id,
            "video_id": unit_id,
            "method": "wilddeepfake",
            "domain": "wilddeepfake",
            "dataset_partition": (
                "wilddeepfake_fake_test"
            ),
            "top_level_folder": (
                "fake_test"
            ),
        }
    )


wild_manifest = pd.DataFrame(
    manifest_rows
)


# ============================================================
# Audit manifest
# ============================================================

duplicate_paths = int(
    wild_manifest[
        "image_path"
    ].duplicated().sum()
)

mixed_label_units = (
    wild_manifest.groupby(
        "sample_unit_id"
    )["label"]
    .nunique()
)

mixed_label_units = mixed_label_units[
    mixed_label_units > 1
]

missing_paths = (
    ~wild_manifest[
        "image_path"
    ].map(
        lambda value: Path(value).exists()
    )
)

if duplicate_paths > 0:
    raise RuntimeError(
        f"Found {duplicate_paths} duplicate image paths."
    )

if len(mixed_label_units) > 0:
    raise RuntimeError(
        f"Found {len(mixed_label_units)} "
        "mixed-label sequence IDs."
    )

if missing_paths.any():
    raise RuntimeError(
        f"{int(missing_paths.sum())} manifest images "
        "do not exist."
    )

observed_labels = set(
    wild_manifest[
        "label"
    ].unique()
)

if observed_labels != {0, 1}:
    raise RuntimeError(
        "WildDeepfake external test must contain "
        "both real and fake classes."
    )


# ============================================================
# Manifest summary
# ============================================================

manifest_summary = (
    wild_manifest.groupby(
        [
            "label",
            "label_name",
        ],
        as_index=False,
    )
    .agg(
        frames=(
            "image_path",
            "size",
        ),
        sequences=(
            "sample_unit_id",
            "nunique",
        ),
    )
)

wild_manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)

print("\n" + "=" * 90)
print("WILDDEEPFAKE MANIFEST READY")
print("=" * 90)

display(
    manifest_summary
)

print("\nDuplicate paths:")
print(duplicate_paths)

print("\nMixed-label sequences:")
print(len(mixed_label_units))

print("\nManifest saved:")
print(MANIFEST_PATH)


# ============================================================
# Optional test-set sampling
# ============================================================

evaluation_manifest = (
    wild_manifest.copy()
)


if (
    FRAMES_PER_UNIT > 0
    or LIMIT_UNITS > 0
):

    print(
        "\nWARNING: subset mode is enabled. "
        "These should not be used as final paper results."
    )

    if hasattr(
        agrfn,
        "sample_frames_per_unit",
    ):

        evaluation_manifest = (
            agrfn.sample_frames_per_unit(
                evaluation_manifest,
                frames_per_unit=(
                    FRAMES_PER_UNIT
                    if FRAMES_PER_UNIT > 0
                    else None
                ),
                limit_units=(
                    LIMIT_UNITS
                    if LIMIT_UNITS > 0
                    else None
                ),
                seed=42,
                desc=(
                    "Sample WildDeepfake external test"
                ),
                balanced_units=True,
            )
        )

    else:

        rng = np.random.default_rng(
            42
        )

        if LIMIT_UNITS > 0:

            selected_units = []

            for label in [0, 1]:

                units = (
                    evaluation_manifest.loc[
                        evaluation_manifest[
                            "label"
                        ] == label,
                        "sample_unit_id",
                    ]
                    .drop_duplicates()
                    .tolist()
                )

                rng.shuffle(
                    units
                )

                selected_units.extend(
                    units[
                        :LIMIT_UNITS
                    ]
                )

            evaluation_manifest = (
                evaluation_manifest[
                    evaluation_manifest[
                        "sample_unit_id"
                    ].isin(
                        selected_units
                    )
                ].copy()
            )

        if FRAMES_PER_UNIT > 0:

            evaluation_manifest = (
                evaluation_manifest.groupby(
                    "sample_unit_id",
                    group_keys=False,
                )
                .head(
                    FRAMES_PER_UNIT
                )
                .reset_index(
                    drop=True
                )
            )


print("\nEvaluation frames:")
print(
    f"{len(evaluation_manifest):,}"
)

print("\nEvaluation sequences:")
print(
    f"{evaluation_manifest['sample_unit_id'].nunique():,}"
)


# ============================================================
# Helpers
# ============================================================

def normalized_name(value):

    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def python_scalar(value):

    if isinstance(
        value,
        (
            str,
            bool,
            int,
            float,
        ),
    ):
        return value

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if torch.is_tensor(value):

        if value.numel() == 1:
            return value.detach().cpu().item()

    return None


def flatten_mapping(
    mapping,
    prefix="",
    depth=0,
    max_depth=3,
):

    output = {}

    if depth > max_depth:
        return output

    if isinstance(
        mapping,
        argparse.Namespace,
    ):
        mapping = vars(
            mapping
        )

    if not isinstance(
        mapping,
        dict,
    ):
        return output

    for key, value in mapping.items():

        full_key = (
            f"{prefix}.{key}"
            if prefix
            else str(key)
        )

        scalar = python_scalar(
            value
        )

        if scalar is not None:
            output[
                full_key
            ] = scalar

        elif isinstance(
            value,
            (
                dict,
                argparse.Namespace,
            ),
        ):

            output.update(
                flatten_mapping(
                    value,
                    prefix=full_key,
                    depth=depth + 1,
                    max_depth=max_depth,
                )
            )

    return output


def load_checkpoint(
    checkpoint_path,
):

    try:
        return torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False,
        )

    except TypeError:

        return torch.load(
            checkpoint_path,
            map_location=device,
        )


# ============================================================
# Recover DF40 decision threshold
# ============================================================

def recover_threshold_from_checkpoint(
    checkpoint,
):

    flattened = flatten_mapping(
        checkpoint
    )

    candidates = []

    preferred_names = [
        "bestdecisionthreshold",
        "validationdecisionthreshold",
        "valdecisionthreshold",
        "bestthreshold",
        "decisionthreshold",
    ]

    for key, value in flattened.items():

        normalized = normalized_name(
            key
        )

        if "uncertainty" in normalized:
            continue

        if "threshold" not in normalized:
            continue

        try:
            numeric_value = float(
                value
            )
        except Exception:
            continue

        if not (
            0.0
            <= numeric_value
            <= 1.0
        ):
            continue

        priority = len(
            preferred_names
        ) + 1

        for index, preferred in enumerate(
            preferred_names
        ):
            if preferred in normalized:
                priority = index
                break

        candidates.append(
            (
                priority,
                key,
                numeric_value,
            )
        )

    if candidates:

        candidates.sort(
            key=lambda item: item[0]
        )

        _, key, value = candidates[0]

        return (
            value,
            f"checkpoint:{key}",
        )

    return None


def recover_threshold_from_csv(
    experiment_dir,
    checkpoint,
):

    preferred_files = [
        experiment_dir
        / "experiment_summary.csv",

        experiment_dir
        / "test_metrics.csv",

        experiment_dir
        / "val_metrics.csv",

        experiment_dir
        / "validation_metrics.csv",

        experiment_dir
        / "training_history.csv",
    ]

    checkpoint_epoch = checkpoint.get(
        "epoch",
        None,
    )

    for csv_path in preferred_files:

        if not csv_path.exists():
            continue

        try:
            table = pd.read_csv(
                csv_path
            )
        except Exception:
            continue

        if len(table) == 0:
            continue

        # For training history, prefer the row matching
        # the checkpoint epoch when possible.
        selected_table = table

        if (
            checkpoint_epoch is not None
            and "epoch" in table.columns
        ):

            epoch_values = pd.to_numeric(
                table[
                    "epoch"
                ],
                errors="coerce",
            )

            matches = table[
                epoch_values
                == float(
                    checkpoint_epoch
                )
            ]

            if len(matches) > 0:
                selected_table = (
                    matches.tail(1)
                )

        for column in selected_table.columns:

            normalized = normalized_name(
                column
            )

            if "threshold" not in normalized:
                continue

            if "uncertainty" in normalized:
                continue

            values = pd.to_numeric(
                selected_table[
                    column
                ],
                errors="coerce",
            ).dropna()

            values = values[
                values.between(
                    0.0,
                    1.0,
                )
            ]

            if len(values) == 0:
                continue

            return (
                float(
                    values.iloc[-1]
                ),
                (
                    f"{csv_path.name}:"
                    f"{column}"
                ),
            )

    return None


def recover_decision_threshold(
    experiment_dir,
    checkpoint,
):

    result = (
        recover_threshold_from_checkpoint(
            checkpoint
        )
    )

    if result is not None:
        return result

    result = recover_threshold_from_csv(
        experiment_dir,
        checkpoint,
    )

    if result is not None:
        return result

    if ALLOW_THRESHOLD_05_FALLBACK:

        print(
            "\nWARNING: validation threshold was "
            "not found. Using 0.5 fallback."
        )

        return (
            0.50,
            "explicit_0.5_fallback",
        )

    raise RuntimeError(
        "Could not recover the DF40 validation-tuned "
        "decision threshold for this checkpoint.\n"
        "The evaluator will not silently tune a new "
        "threshold on WildDeepfake."
    )


# ============================================================
# Recover score inversion / direction
# ============================================================

def parse_direction_value(
    value,
):

    if isinstance(
        value,
        bool,
    ):
        return value

    normalized = str(
        value
    ).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
        "inverted",
        "invert",
        "inverse",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
        "normal",
    }:
        return False

    return None


def recover_score_direction_from_checkpoint(
    checkpoint,
):

    flattened = flatten_mapping(
        checkpoint
    )

    # Explicit resolved inversion keys first.
    for key, value in flattened.items():

        normalized = normalized_name(
            key
        )

        if (
            "scoreinverted" in normalized
            or "invertscores" in normalized
            or "scoreflip" in normalized
        ):

            parsed = parse_direction_value(
                value
            )

            if parsed is not None:
                return (
                    parsed,
                    f"checkpoint:{key}",
                )

    # Explicit normal/inverted string direction.
    for key, value in flattened.items():

        normalized = normalized_name(
            key
        )

        if "scoredirection" not in normalized:
            continue

        parsed = parse_direction_value(
            value
        )

        if parsed is not None:
            return (
                parsed,
                f"checkpoint:{key}",
            )

    return None


def recover_score_direction_from_files(
    experiment_dir,
):

    preferred_files = [
        experiment_dir
        / "experiment_summary.csv",

        experiment_dir
        / "test_frame_predictions.csv",

        experiment_dir
        / "val_frame_predictions.csv",

        experiment_dir
        / "validation_frame_predictions.csv",

        experiment_dir
        / "test_metrics.csv",

        experiment_dir
        / "val_metrics.csv",
    ]

    for csv_path in preferred_files:

        if not csv_path.exists():
            continue

        try:
            table = pd.read_csv(
                csv_path
            )
        except Exception:
            continue

        if len(table) == 0:
            continue

        for column in table.columns:

            normalized = normalized_name(
                column
            )

            if (
                "scoreinverted" not in normalized
                and "scoredirection" not in normalized
            ):
                continue

            values = (
                table[
                    column
                ]
                .dropna()
                .tolist()
            )

            for value in values:

                parsed = parse_direction_value(
                    value
                )

                if parsed is not None:

                    return (
                        parsed,
                        (
                            f"{csv_path.name}:"
                            f"{column}"
                        ),
                    )

    return None


def recover_score_direction(
    experiment_dir,
    checkpoint,
):

    result = (
        recover_score_direction_from_checkpoint(
            checkpoint
        )
    )

    if result is not None:
        return result

    result = (
        recover_score_direction_from_files(
            experiment_dir
        )
    )

    if result is not None:
        return result

    # If there is no evidence of auto-flipping,
    # normal direction is the safest explicit default.
    checkpoint_args = checkpoint.get(
        "args",
        {},
    )

    if isinstance(
        checkpoint_args,
        argparse.Namespace,
    ):
        checkpoint_args = vars(
            checkpoint_args
        )

    configured_direction = str(
        checkpoint_args.get(
            "score_direction",
            "normal",
        )
    ).strip().lower()

    if configured_direction == "normal":
        return (
            False,
            "checkpoint_args:normal",
        )

    if configured_direction == "inverted":
        return (
            True,
            "checkpoint_args:inverted",
        )

    raise RuntimeError(
        "The checkpoint used automatic score direction, "
        "but the resolved DF40 score direction could "
        "not be recovered."
    )


# ============================================================
# Recover model args
# ============================================================

def checkpoint_model_args(
    checkpoint,
):

    stored_args = checkpoint.get(
        "args",
        {},
    )

    if isinstance(
        stored_args,
        argparse.Namespace,
    ):
        stored_args = vars(
            stored_args
        )

    if not isinstance(
        stored_args,
        dict,
    ):
        raise RuntimeError(
            "Checkpoint args are not stored as "
            "a dictionary or Namespace."
        )

    args = SimpleNamespace(
        **stored_args
    )

    defaults = {
        "fusion_mode": "late",
        "streams": (
            "rgb,fingerprint,jpeg,freq"
        ),
        "feat_dim": 128,
        "controller_dim": 128,
        "routing_temperature": 1.0,
        "film_scale": 0.10,
        "router_heads": 4,
        "router_layers": 1,
        "router_dropout": 0.10,
        "stream_logit_residual_weight": 0.25,
        "use_prototypes": False,
        "prototype_temperature": 0.10,
        "use_uncertainty": False,
        "image_size": 224,
        "batch_size": 8,
        "num_workers": 0,
        "data_root": None,
    }

    for name, value in defaults.items():

        if not hasattr(
            args,
            name,
        ):
            setattr(
                args,
                name,
                value,
            )

    return args


def checkpoint_stream_names(
    checkpoint,
    args,
):

    stream_names = checkpoint.get(
        "stream_names",
        None,
    )

    if stream_names is None:

        stream_names = [
            value.strip()
            for value in str(
                args.streams
            ).split(",")
            if value.strip()
        ]

    return list(
        stream_names
    )


# ============================================================
# Metrics
# ============================================================

def calculate_eer(
    labels,
    fake_probabilities,
):

    labels = np.asarray(
        labels,
        dtype=int,
    )

    scores = np.asarray(
        fake_probabilities,
        dtype=float,
    )

    if len(
        np.unique(
            labels
        )
    ) < 2:
        return np.nan

    fpr, tpr, _ = roc_curve(
        labels,
        scores,
        pos_label=1,
    )

    fnr = (
        1.0
        - tpr
    )

    index = int(
        np.nanargmin(
            np.abs(
                fpr
                - fnr
            )
        )
    )

    return float(
        (
            fpr[index]
            + fnr[index]
        )
        / 2.0
    )


def calculate_metrics(
    dataframe,
    level,
    experiment_name,
    fusion_mode,
    decision_threshold,
    score_inverted,
):

    labels = (
        dataframe[
            "label"
        ]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        dataframe[
            "fake_prob"
        ]
        .astype(float)
        .to_numpy()
    )

    predicted = (
        probabilities
        >= decision_threshold
    ).astype(int)

    if len(
        np.unique(
            labels
        )
    ) != 2:
        raise RuntimeError(
            f"{level} evaluation does not contain "
            "both classes."
        )

    tn, fp, fn, tp = (
        confusion_matrix(
            labels,
            predicted,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )

    result = {
        "Experiment": experiment_name,
        "Fusion Mode": fusion_mode,
        "Dataset": "WildDeepfake",
        "Training Dataset": "DF40",
        "Evaluation Type": (
            "Zero-shot cross-dataset"
        ),
        "Level": level,
        "Samples": int(
            len(dataframe)
        ),
        "Decision Threshold": float(
            decision_threshold
        ),
        "Score Inverted": bool(
            score_inverted
        ),
        "Accuracy": float(
            accuracy_score(
                labels,
                predicted,
            )
        ),
        "Balanced Accuracy": float(
            balanced_accuracy_score(
                labels,
                predicted,
            )
        ),
        "Precision (0)": float(
            precision_score(
                labels,
                predicted,
                pos_label=0,
                zero_division=0,
            )
        ),
        "Recall (0)": float(
            recall_score(
                labels,
                predicted,
                pos_label=0,
                zero_division=0,
            )
        ),
        "F1 (0)": float(
            f1_score(
                labels,
                predicted,
                pos_label=0,
                zero_division=0,
            )
        ),
        "Precision (1)": float(
            precision_score(
                labels,
                predicted,
                pos_label=1,
                zero_division=0,
            )
        ),
        "Recall (1)": float(
            recall_score(
                labels,
                predicted,
                pos_label=1,
                zero_division=0,
            )
        ),
        "F1 (1)": float(
            f1_score(
                labels,
                predicted,
                pos_label=1,
                zero_division=0,
            )
        ),
        "Macro F1": float(
            f1_score(
                labels,
                predicted,
                average="macro",
                zero_division=0,
            )
        ),
        "AUCROC": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "AUPRC": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "EER": float(
            calculate_eer(
                labels,
                probabilities,
            )
        ),
        "True Negatives": int(
            tn
        ),
        "False Positives": int(
            fp
        ),
        "False Negatives": int(
            fn
        ),
        "True Positives": int(
            tp
        ),
        "Real Support": int(
            (labels == 0).sum()
        ),
        "Fake Support": int(
            (labels == 1).sum()
        ),
        "Mean Fake Probability": float(
            np.mean(
                probabilities
            )
        ),
    }

    if "uncertainty" in dataframe.columns:

        uncertainty = pd.to_numeric(
            dataframe[
                "uncertainty"
            ],
            errors="coerce",
        )

        valid_uncertainty = (
            uncertainty.dropna()
        )

        if len(
            valid_uncertainty
        ) > 0:

            result[
                "Mean Uncertainty"
            ] = float(
                valid_uncertainty.mean()
            )

            result[
                "Median Uncertainty"
            ] = float(
                valid_uncertainty.median()
            )

            result[
                "Uncertain Rate"
            ] = float(
                (
                    valid_uncertainty
                    >= UNCERTAINTY_THRESHOLD
                ).mean()
            )

    return result


# ============================================================
# Sequence-level aggregation
# ============================================================

def aggregate_sequences(
    predictions,
):

    aggregation = {
        "label": "first",
        "fake_prob": "mean",
        "raw_fake_prob": "mean",
        "uncertainty": "mean",
        "label_name": "first",
        "method": "first",
        "domain": "first",
        "dataset_partition": "first",
        "image_path": "count",
    }

    aggregation = {
        column: operation
        for column, operation
        in aggregation.items()
        if column
        in predictions.columns
    }

    sequences = (
        predictions.groupby(
            "sample_unit_id",
            as_index=False,
        )
        .agg(
            aggregation
        )
    )

    if "image_path" in sequences.columns:

        sequences = sequences.rename(
            columns={
                "image_path": (
                    "num_frames"
                )
            }
        )

    return sequences


# ============================================================
# Discover experiments
# ============================================================

if EXPERIMENTS is None:

    discovered = []

    for experiment_dir in sorted(
        ABLATION_ROOT.glob(
            "E*"
        )
    ):

        if not experiment_dir.is_dir():
            continue

        if (
            experiment_dir
            / "best_checkpoint.pt"
        ).exists():

            discovered.append(
                experiment_dir.name
            )

    EXPERIMENTS = discovered


if not EXPERIMENTS:

    raise RuntimeError(
        "No experiments with best_checkpoint.pt "
        "were found."
    )


print("\n" + "=" * 90)
print("MODELS SELECTED")
print("=" * 90)

for experiment_name in EXPERIMENTS:
    print(
        " ",
        experiment_name,
    )


# ============================================================
# Evaluation loop
# ============================================================

all_metric_rows = []
status_rows = []


for model_index, experiment_name in enumerate(
    EXPERIMENTS,
    start=1,
):

    print("\n\n")
    print("#" * 90)
    print(
        f"[{model_index}/{len(EXPERIMENTS)}] "
        f"{experiment_name}"
    )
    print("#" * 90)

    experiment_dir = (
        ABLATION_ROOT
        / experiment_name
    )

    checkpoint_path = (
        experiment_dir
        / "best_checkpoint.pt"
    )

    wild_output_dir = (
        experiment_dir
        / "wilddeepfake_test"
    )

    wild_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    started_at = (
        datetime.now().isoformat(
            timespec="seconds"
        )
    )

    try:

        if not checkpoint_path.exists():
            raise FileNotFoundError(
                checkpoint_path
            )


        # ----------------------------------------------------
        # Load checkpoint
        # ----------------------------------------------------

        print("\nLoading checkpoint:")
        print(checkpoint_path)

        checkpoint = load_checkpoint(
            checkpoint_path
        )


        # ----------------------------------------------------
        # Recover model configuration
        # ----------------------------------------------------

        args = checkpoint_model_args(
            checkpoint
        )

        stream_names = (
            checkpoint_stream_names(
                checkpoint,
                args,
            )
        )


        # ----------------------------------------------------
        # Recover DF40 decision policy
        # ----------------------------------------------------

        (
            decision_threshold,
            threshold_source,
        ) = recover_decision_threshold(
            experiment_dir,
            checkpoint,
        )

        (
            invert_scores,
            score_direction_source,
        ) = recover_score_direction(
            experiment_dir,
            checkpoint,
        )

        print("\nFusion mode:")
        print(args.fusion_mode)

        print("\nStreams:")
        print(stream_names)

        print("\nDF40 decision threshold:")
        print(decision_threshold)

        print("\nThreshold source:")
        print(threshold_source)

        print("\nScore inverted:")
        print(invert_scores)

        print("\nScore-direction source:")
        print(score_direction_source)


        # ----------------------------------------------------
        # Build model
        # ----------------------------------------------------

        model = agrfn.build_model(
            args,
            stream_names,
        ).to(
            device
        )

        state_dict = checkpoint.get(
            "model",
            None,
        )

        if state_dict is None:
            state_dict = checkpoint.get(
                "model_state_dict",
                None,
            )

        if state_dict is None:
            raise RuntimeError(
                "Checkpoint contains no model state."
            )

        model.load_state_dict(
            state_dict,
            strict=True,
        )

        model.eval()


        # ----------------------------------------------------
        # Build WildDeepfake dataset
        # ----------------------------------------------------

        batch_size = (
            int(BATCH_SIZE)
            if BATCH_SIZE is not None
            else int(
                args.batch_size
            )
        )

        num_workers = (
            int(NUM_WORKERS)
            if NUM_WORKERS is not None
            else int(
                args.num_workers
            )
        )

        dataset = agrfn.GRFNDataset(
            evaluation_manifest,
            image_size=int(
                args.image_size
            ),
            train=False,
            train_aug=False,
        )

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=(
                device.type
                == "cuda"
            ),
        )


        print("\nWildDeepfake evaluation:")
        print(
            f"  Frames: "
            f"{len(evaluation_manifest):,}"
        )

        print(
            f"  Sequences: "
            f"{evaluation_manifest['sample_unit_id'].nunique():,}"
        )

        print(
            f"  Batch size: "
            f"{batch_size}"
        )


        # ----------------------------------------------------
        # Inference
        # ----------------------------------------------------

        if device.type == "cuda":
            torch.cuda.synchronize()

        start_time = (
            time.perf_counter()
        )

        with torch.no_grad():

            predictions = (
                agrfn.predict_loader(
                    model,
                    loader,
                    args,
                    device,
                    split_name=(
                        "wilddeepfake_test"
                    ),
                )
            )

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )


        # ----------------------------------------------------
        # Apply DF40 score direction
        # ----------------------------------------------------

        predictions = (
            predictions.copy()
        )

        predictions[
            "raw_fake_prob"
        ] = predictions[
            "fake_prob"
        ]

        if invert_scores:

            predictions[
                "fake_prob"
            ] = (
                1.0
                - predictions[
                    "fake_prob"
                ]
            )


        predictions[
            "predicted_label"
        ] = (
            predictions[
                "fake_prob"
            ]
            >= decision_threshold
        ).astype(int)


        if "uncertainty" in predictions.columns:

            predictions[
                "is_uncertain"
            ] = (
                predictions[
                    "uncertainty"
                ]
                >= UNCERTAINTY_THRESHOLD
            ).astype(int)


        # ----------------------------------------------------
        # Validate prediction alignment
        # ----------------------------------------------------

        if len(predictions) != len(
            evaluation_manifest
        ):

            raise RuntimeError(
                "Prediction row count does not match "
                "the WildDeepfake manifest.\n"
                f"Predictions: {len(predictions)}\n"
                f"Manifest: {len(evaluation_manifest)}"
            )


        if not predictions[
            "fake_prob"
        ].between(
            0.0,
            1.0,
        ).all():

            raise RuntimeError(
                "WildDeepfake probabilities "
                "are outside [0, 1]."
            )


        # ----------------------------------------------------
        # Sequence aggregation
        # ----------------------------------------------------

        sequence_predictions = (
            aggregate_sequences(
                predictions
            )
        )

        sequence_predictions[
            "predicted_label"
        ] = (
            sequence_predictions[
                "fake_prob"
            ]
            >= decision_threshold
        ).astype(int)


        if (
            "uncertainty"
            in sequence_predictions.columns
        ):

            sequence_predictions[
                "is_uncertain"
            ] = (
                sequence_predictions[
                    "uncertainty"
                ]
                >= UNCERTAINTY_THRESHOLD
            ).astype(int)


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        frame_metrics = (
            calculate_metrics(
                predictions,
                level="Frame",
                experiment_name=(
                    experiment_name
                ),
                fusion_mode=str(
                    args.fusion_mode
                ),
                decision_threshold=(
                    decision_threshold
                ),
                score_inverted=(
                    invert_scores
                ),
            )
        )

        sequence_metrics = (
            calculate_metrics(
                sequence_predictions,
                level="Sequence",
                experiment_name=(
                    experiment_name
                ),
                fusion_mode=str(
                    args.fusion_mode
                ),
                decision_threshold=(
                    decision_threshold
                ),
                score_inverted=(
                    invert_scores
                ),
            )
        )

        metrics_table = pd.DataFrame(
            [
                frame_metrics,
                sequence_metrics,
            ]
        )

        all_metric_rows.extend(
            [
                frame_metrics,
                sequence_metrics,
            ]
        )


        # ----------------------------------------------------
        # Inference timing
        # ----------------------------------------------------

        timing_table = pd.DataFrame(
            [
                {
                    "Experiment": (
                        experiment_name
                    ),
                    "Dataset": (
                        "WildDeepfake"
                    ),
                    "Training Dataset": (
                        "DF40"
                    ),
                    "Fusion Mode": str(
                        args.fusion_mode
                    ),
                    "Device": str(
                        device
                    ),
                    "Image Size": int(
                        args.image_size
                    ),
                    "Batch Size": int(
                        batch_size
                    ),
                    "Frames Scored": int(
                        len(
                            predictions
                        )
                    ),
                    "Sequences Scored": int(
                        len(
                            sequence_predictions
                        )
                    ),
                    "Total Seconds": float(
                        elapsed_seconds
                    ),
                    "Time Per Frame (ms)": float(
                        elapsed_seconds
                        / max(
                            len(
                                predictions
                            ),
                            1,
                        )
                        * 1000.0
                    ),
                    "FPS": float(
                        len(
                            predictions
                        )
                        / max(
                            elapsed_seconds,
                            1e-8,
                        )
                    ),
                }
            ]
        )


        # ----------------------------------------------------
        # Save outputs
        # ----------------------------------------------------

        predictions.to_csv(
            wild_output_dir
            / "wilddeepfake_frame_predictions.csv",
            index=False,
        )

        sequence_predictions.to_csv(
            wild_output_dir
            / "wilddeepfake_sequence_predictions.csv",
            index=False,
        )

        metrics_table.to_csv(
            wild_output_dir
            / "wilddeepfake_metrics.csv",
            index=False,
        )

        timing_table.to_csv(
            wild_output_dir
            / "wilddeepfake_inference_time.csv",
            index=False,
        )

        evaluation_manifest.to_csv(
            wild_output_dir
            / "wilddeepfake_evaluation_manifest.csv",
            index=False,
        )


        # ----------------------------------------------------
        # Save architecture-specific diagnostics
        # ----------------------------------------------------

        diagnostic_files = []

        if hasattr(
            agrfn,
            "save_diagnostic_csvs",
        ):

            try:

                diagnostic_result = (
                    agrfn.save_diagnostic_csvs(
                        predictions,
                        wild_output_dir,
                    )
                )

                if diagnostic_result:
                    diagnostic_files = [
                        str(value)
                        for value
                        in diagnostic_result
                    ]

            except Exception as diagnostic_error:

                print(
                    "\nWARNING: diagnostic CSV "
                    "generation failed:"
                )

                print(
                    repr(
                        diagnostic_error
                    )
                )


        # ----------------------------------------------------
        # Run configuration
        # ----------------------------------------------------

        run_configuration = {
            "experiment_name": (
                experiment_name
            ),
            "evaluation_dataset": (
                "WildDeepfake"
            ),
            "training_dataset": (
                "DF40"
            ),
            "evaluation_type": (
                "zero-shot cross-dataset"
            ),
            "checkpoint": str(
                checkpoint_path
            ),
            "training_script": str(
                TRAINING_SCRIPT
            ),
            "wilddeepfake_manifest": str(
                MANIFEST_PATH
            ),
            "fusion_mode": str(
                args.fusion_mode
            ),
            "stream_names": (
                stream_names
            ),
            "use_prototypes": bool(
                getattr(
                    args,
                    "use_prototypes",
                    False,
                )
            ),
            "use_uncertainty": bool(
                getattr(
                    args,
                    "use_uncertainty",
                    False,
                )
            ),
            "decision_threshold": float(
                decision_threshold
            ),
            "threshold_source": (
                threshold_source
            ),
            "score_inverted": bool(
                invert_scores
            ),
            "score_direction_source": (
                score_direction_source
            ),
            "uncertainty_threshold": float(
                UNCERTAINTY_THRESHOLD
            ),
            "frames_per_unit": int(
                FRAMES_PER_UNIT
            ),
            "limit_units": int(
                LIMIT_UNITS
            ),
            "frames_evaluated": int(
                len(
                    predictions
                )
            ),
            "sequences_evaluated": int(
                len(
                    sequence_predictions
                )
            ),
            "device": str(
                device
            ),
            "batch_size": int(
                batch_size
            ),
            "image_size": int(
                args.image_size
            ),
            "diagnostic_files": (
                diagnostic_files
            ),
        }


        with open(
            wild_output_dir
            / "wilddeepfake_run_configuration.json",
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                run_configuration,
                handle,
                indent=2,
            )


        # ----------------------------------------------------
        # Display results
        # ----------------------------------------------------

        print("\n" + "=" * 90)
        print(
            f"WILDDEEPFAKE RESULTS: "
            f"{experiment_name}"
        )
        print("=" * 90)

        display(
            metrics_table
        )

        print("\nInference timing:")
        display(
            timing_table
        )

        print("\nOutput directory:")
        print(
            wild_output_dir
        )


        # ----------------------------------------------------
        # Status
        # ----------------------------------------------------

        status_rows.append(
            {
                "experiment_name": (
                    experiment_name
                ),
                "status": "completed",
                "fusion_mode": str(
                    args.fusion_mode
                ),
                "checkpoint": str(
                    checkpoint_path
                ),
                "decision_threshold": float(
                    decision_threshold
                ),
                "score_inverted": bool(
                    invert_scores
                ),
                "frames": int(
                    len(
                        predictions
                    )
                ),
                "sequences": int(
                    len(
                        sequence_predictions
                    )
                ),
                "started_at": (
                    started_at
                ),
                "finished_at": (
                    datetime.now()
                    .isoformat(
                        timespec="seconds"
                    )
                ),
                "error": "",
            }
        )


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del model
        del dataset
        del loader
        del checkpoint

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    except Exception as error:

        print("\n" + "!" * 90)
        print(
            f"FAILED: {experiment_name}"
        )
        print("!" * 90)

        print(
            repr(
                error
            )
        )

        traceback.print_exc()

        status_rows.append(
            {
                "experiment_name": (
                    experiment_name
                ),
                "status": "failed",
                "fusion_mode": "",
                "checkpoint": str(
                    checkpoint_path
                ),
                "decision_threshold": np.nan,
                "score_inverted": np.nan,
                "frames": 0,
                "sequences": 0,
                "started_at": (
                    started_at
                ),
                "finished_at": (
                    datetime.now()
                    .isoformat(
                        timespec="seconds"
                    )
                ),
                "error": repr(
                    error
                ),
            }
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if not CONTINUE_ON_MODEL_ERROR:
            raise


    # --------------------------------------------------------
    # Continuously save status and accumulated results
    # --------------------------------------------------------

    pd.DataFrame(
        status_rows
    ).to_csv(
        STATUS_PATH,
        index=False,
    )

    if all_metric_rows:

        pd.DataFrame(
            all_metric_rows
        ).to_csv(
            COMBINED_SUMMARY_PATH,
            index=False,
        )


# ============================================================
# Final combined summary
# ============================================================

status_table = pd.DataFrame(
    status_rows
)

summary_table = pd.DataFrame(
    all_metric_rows
)


print("\n\n")
print("=" * 90)
print("WILDDEEPFAKE CROSS-DATASET EVALUATION FINISHED")
print("=" * 90)

print("\nEvaluation status:")
display(
    status_table
)


if len(summary_table) > 0:

    display_columns = [
        "Experiment",
        "Fusion Mode",
        "Level",
        "Samples",
        "Decision Threshold",
        "Accuracy",
        "Balanced Accuracy",
        "Precision (0)",
        "Recall (0)",
        "F1 (0)",
        "Precision (1)",
        "Recall (1)",
        "F1 (1)",
        "Macro F1",
        "AUCROC",
        "AUPRC",
        "EER",
    ]

    display_columns = [
        column
        for column in display_columns
        if column
        in summary_table.columns
    ]

    print("\nCombined WildDeepfake results:")

    display(
        summary_table[
            display_columns
        ]
    )


print("\nCombined results saved to:")
print(
    COMBINED_SUMMARY_PATH
)

print("\nStatus saved to:")
print(
    STATUS_PATH
)


completed_count = int(
    (
        status_table[
            "status"
        ]
        == "completed"
    ).sum()
)

failed_count = int(
    (
        status_table[
            "status"
        ]
        == "failed"
    ).sum()
)

print("\nCompleted models:")
print(
    completed_count
)

print("\nFailed models:")
print(
    failed_count
)


if failed_count == 0:

    print(
        "\nPASS: all available DF40 checkpoints "
        "completed zero-shot WildDeepfake evaluation."
    )

else:

    print(
        "\nSome models failed. Check "
        "wilddeepfake_evaluation_status.csv "
        "for the exact reason."
    )

DF40 -> WILDDEEPFAKE ZERO-SHOT CROSS-DATASET EVALUATION

Device:
cuda

GPU:
NVIDIA A100 80GB PCIe

Real test root:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test

Fake test root:
/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test

No WildDeepfake training or threshold tuning will be performed.

PASS: A-GRFN module imported.


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



WildDeepfake images discovered:
  Real: 58,659
  Fake: 107,004
  Total: 165,663

WILDDEEPFAKE MANIFEST READY


,label,label_name,frames,sequences
0,0,real,58659,396
1,1,fake,107004,411



Duplicate paths:
0

Mixed-label sequences:
0

Manifest saved:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest.csv

Evaluation frames:
165,663

Evaluation sequences:
807

MODELS SELECTED
  E0_late_fusion



##########################################################################################
[1/1] E0_late_fusion
##########################################################################################

Loading checkpoint:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_ablation_suite/E0_late_fusion/best_checkpoint.pt

Fusion mode:
late

Streams:
['rgb', 'fingerprint', 'jpeg', 'freq']

DF40 decision threshold:
0.6306639599999999

Threshold source:
checkpoint:args.decision_threshold

Score inverted:
False

Score-direction source:
experiment_summary.csv:Score Inverted For Eval

WildDeepfake evaluation:
  Frames: 165,663
  Sequences: 807
  Batch size: 16


Predict wilddeepfake_test: 100%|██████████| 10354/10354 [1:54:41<00:00,  1.50it/s] 


... (5 outputs left out)

,experiment_name,status,fusion_mode,checkpoint,decision_threshold,score_inverted,frames,sequences,started_at,finished_at,error
0,E0_late_fusion,completed,late,/home/jovyan/Deepfake_Low Resource_Low Resolut...,0.630664,False,165663,807,2026-08-07T05:55:37,2026-08-07T07:50:39,



Combined WildDeepfake results:


,Experiment,Fusion Mode,Level,Samples,Decision Threshold,Accuracy,Balanced Accuracy,Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Macro F1,AUCROC,AUPRC,EER
0,E0_late_fusion,late,Frame,165663,0.630664,0.619975,0.551818,0.448397,0.318263,0.372285,0.677573,0.785373,0.727501,0.549893,0.561061,0.667174,0.449158
1,E0_late_fusion,late,Sequence,807,0.630664,0.587361,0.584154,0.619772,0.411616,0.494689,0.571691,0.756691,0.651309,0.572999,0.609323,0.575002,0.405229



Combined results saved to:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_cross_dataset_summary.csv

Status saved to:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_evaluation_status.csv

Completed models:
1

Failed models:
0

PASS: all available DF40 checkpoints completed zero-shot WildDeepfake evaluation.


In [30]:
!pip install torch scikit-learn pillow

  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)


In [8]:
PROJECT_ROOT="/home/jovyan/Deepfake_Low Resource_Low Resolution"
WILD_MANIFEST="$PROJECT_ROOT/outputs_wilddeepfake/wilddeepfake_test_manifest.csv"

# Edit these if your best checkpoints are in different folders.
GRFN_CKPT="$PROJECT_ROOT/outputs_grfn_df40_indomain/best_checkpoint.pt"
FREQ_CKPT="$PROJECT_ROOT/outputs_freqdebias_df40_indomain/best_checkpoint.pt"
GEND_CKPT="$PROJECT_ROOT/outputs_gend_reimpl_df40_indomain/best_checkpoint.pt"
DFDHR_CKPT="$PROJECT_ROOT/DFD-HR/ckpt_best.pth"

# cd "$PROJECT_ROOT"

!python freqdebias_ffpp.py \
  --metadata_csv "$WILD_MANIFEST" \
  --out_dir "$PROJECT_ROOT/outputs_freqdebias_wilddeepfake_eval" \
  --checkpoint "$FREQ_CKPT" \
  --eval_only \
  --split test \
  --batch_size 16 \
  --image_size 256 \
  --num_workers 0

Ignoring notebook/system args: ['--checkpoint', '/outputs_freqdebias_df40_indomain/best_checkpoint.pt', '--eval_only', '--split', 'test']

FreqDebias FF++ regular experiment
Device: cuda
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval
Traceback (most recent call last):
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 1007, in <module>
    main()
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 873, in main
    df = load_ffpp_metadata(args.metadata_csv)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 128, in load_ffpp_metadata
    raise FileNotFoundError(f"Missing metadata CSV: {metadata_csv}")
FileNotFoundError: Missing metadata CSV: /outputs_wilddeepfake/wilddeepfake_test_manifest.csv


In [12]:
# ============================================================
# Robust FreqDebias DF40 checkpoint -> WildDeepfake evaluation
# Fixes empty split filtering issue.
# ============================================================

from pathlib import Path
import importlib.util
import time
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

# -----------------------------
# Paths / settings
# -----------------------------
PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

METADATA_CSV = PROJECT_ROOT / "outputs_wilddeepfake" / "wilddeepfake_test_manifest.csv"
SCRIPT_PATH = PROJECT_ROOT / "freqdebias_ffpp.py"

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "outputs_freqdebias_df40_indomain" / "best_checkpoint.pt",
    PROJECT_ROOT / "outputs_freqdebias_df40_indomain" / "freqdebias_best.pt",
    PROJECT_ROOT / "outputs_freqdebias_df40_indomain" / "latest_checkpoint.pt",
    PROJECT_ROOT / "outputs_freqdebias_df40_indomain" / "freqdebias_latest.pt",
]

OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
IMAGE_SIZE = 256
NUM_WORKERS = 0
THRESHOLD = 0.5

# -----------------------------
# Resolve checkpoint
# -----------------------------
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"WildDeepfake manifest not found: {METADATA_CSV}")

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"FreqDebias script not found: {SCRIPT_PATH}")

checkpoint = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if checkpoint is None:
    found = sorted(PROJECT_ROOT.rglob("*freq*.pt")) + sorted(PROJECT_ROOT.rglob("*freq*.pth"))
    raise FileNotFoundError(
        "Could not find FreqDebias checkpoint.\n\nChecked:\n"
        + "\n".join(str(p) for p in CHECKPOINT_CANDIDATES)
        + "\n\nPossible files found:\n"
        + "\n".join(str(p) for p in found[:50])
    )

print("Manifest:", METADATA_CSV)
print("Checkpoint:", checkpoint)
print("Script:", SCRIPT_PATH)
print("Output dir:", OUT_DIR)

# -----------------------------
# Import FreqDebias code
# -----------------------------
def import_module_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

freq_mod = import_module_from_path("freqdebias_module", SCRIPT_PATH)

# -----------------------------
# Load and normalize manifest
# -----------------------------
raw_df = pd.read_csv(METADATA_CSV)

print("\nRaw manifest rows:", len(raw_df))
print("Raw manifest columns:")
print(list(raw_df.columns))

if len(raw_df) == 0:
    raise ValueError(f"The manifest file itself has zero rows: {METADATA_CSV}")

path_col = None
for c in ["frame_path", "image_path", "path", "filepath", "file_path"]:
    if c in raw_df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"No path column found. Columns: {list(raw_df.columns)}")

if "label" not in raw_df.columns:
    raise ValueError("Manifest must contain label column with 0=real and 1=fake.")

df = raw_df.copy()
df["frame_path"] = df[path_col].astype(str)
df["image_path"] = df[path_col].astype(str)
df["label"] = df["label"].astype(int)

# Show split values before filtering.
if "split" in df.columns:
    print("\nRaw split counts:")
    display(df["split"].astype(str).value_counts(dropna=False).to_frame("count"))

    split_lower = df["split"].astype(str).str.lower().str.strip()
    test_mask = split_lower.eq("test")

    if test_mask.sum() > 0:
        df = df[test_mask].reset_index(drop=True)
        print(f"\nFiltered to split == test: {len(df)} rows")
    else:
        print("\nWARNING: No rows have split exactly equal to 'test'.")
        print("Because this file is already wilddeepfake_test_manifest.csv, using all rows as test.")
        df["split"] = "test"
else:
    print("\nNo split column found. Treating all rows as test.")
    df["split"] = "test"

if len(df) == 0:
    raise ValueError(
        "After manifest normalization, there are still zero rows. "
        "Open the CSV and check whether it has image paths and labels."
    )

# Standard grouping columns.
if "video_id" not in df.columns:
    if "sample_unit_id" in df.columns:
        df["video_id"] = df["sample_unit_id"].astype(str)
    elif "group_id" in df.columns:
        df["video_id"] = df["group_id"].astype(str)
    elif "clip_id" in df.columns:
        df["video_id"] = df["clip_id"].astype(str)
    else:
        df["video_id"] = df["image_path"].map(lambda p: str(Path(p).parent))

if "sample_unit_id" not in df.columns:
    df["sample_unit_id"] = df["video_id"].astype(str)

if "group_id" not in df.columns:
    df["group_id"] = df["video_id"].astype(str)

if "method" not in df.columns:
    df["method"] = "wilddeepfake"

if "domain" not in df.columns:
    df["domain"] = "wild"

# Verify image files.
missing = [p for p in df["image_path"].tolist() if not Path(p).exists()]
print("\nRows after normalization:", len(df))
print("Missing image files:", len(missing))

if missing:
    print("First missing image:", missing[0])
    raise FileNotFoundError(f"Missing {len(missing)} image files. First missing: {missing[0]}")

print("\nCounts by label:")
display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))

print("\nCounts by split and label:")
display(pd.crosstab(df["split"], df["label"]))

print("\nUnique videos/groups by label:")
display(df.groupby("label")["video_id"].nunique().rename("unique_video_id_count").reset_index())

# -----------------------------
# Metrics
# -----------------------------
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "Decision Threshold": float(threshold),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
    }

    if len(np.unique(y_true)) >= 2:
        out["Balanced Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
        out["AUPRC"] = float(average_precision_score(y_true, y_prob))
        out["AUCROC"] = float(roc_auc_score(y_true, y_prob))
    else:
        out["Balanced Accuracy"] = np.nan
        out["AUPRC"] = np.nan
        out["AUCROC"] = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    out["Precision (0)"] = float(precision[0])
    out["Recall (0)"] = float(recall[0])
    out["F1 (0)"] = float(f1[0])
    out["Precision (1)"] = float(precision[1])
    out["Recall (1)"] = float(recall[1])
    out["F1 (1)"] = float(f1[1])
    out["Real Support"] = int(support[0])
    out["Fake Support"] = int(support[1])
    out["Predicted Real Count"] = int((y_pred == 0).sum())
    out["Predicted Fake Count"] = int((y_pred == 1).sum())
    out["Fake Detection Rate / Recall(1)"] = float(recall[1])
    out["Missed Fake Rate"] = float(1.0 - recall[1]) if support[1] > 0 else np.nan
    out["Mean Fake Probability"] = float(y_prob.mean())
    return out

# -----------------------------
# Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

ckpt = torch.load(checkpoint, map_location=device)

model = freq_mod.FreqDebiasNet(pretrained=False).to(device)
state = ckpt.get("model", ckpt.get("state_dict", ckpt))
model.load_state_dict(state, strict=True)
model.eval()

# -----------------------------
# Dataset / loader
# -----------------------------
dataset = freq_mod.FreqDebiasDataset(
    df,
    image_size=IMAGE_SIZE,
    train=False,
    use_fo_mixup=False,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("\nDataset length:", len(dataset))
print("Loader batches:", len(loader))

if len(dataset) == 0:
    raise ValueError("Dataset length is zero. The manifest normalization failed.")

# -----------------------------
# Predict
# -----------------------------
rows = []
start = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="FreqDebias WildDeepfake", dynamic_ncols=True):
        x = batch["x_t"].to(device, non_blocking=True)
        out = model(x, labels=None)
        prob = F.softmax(out["logits"], dim=1)[:, 1].detach().cpu().numpy()

        labels = batch["label"].detach().cpu().numpy()

        for i in range(len(prob)):
            image_path = str(batch["image_path"][i])
            video_id = str(batch["video_id"][i]) if "video_id" in batch else str(Path(image_path).parent)
            rows.append({
                "image_path": image_path,
                "video_id": video_id,
                "label": int(labels[i]),
                "fake_prob": float(prob[i]),
                "pred_label": int(prob[i] >= THRESHOLD),
                "method": "wilddeepfake",
            })

elapsed = time.time() - start

pred_df = pd.DataFrame(rows)

if len(pred_df) == 0:
    raise ValueError("Prediction DataFrame is empty even though dataset was non-empty.")

# -----------------------------
# Video-level aggregation
# -----------------------------
video_df = pred_df.groupby("video_id").agg(
    label=("label", "first"),
    fake_prob=("fake_prob", "mean"),
    num_frames=("fake_prob", "size"),
).reset_index()

video_df["pred_label"] = (video_df["fake_prob"] >= THRESHOLD).astype(int)

# -----------------------------
# Save outputs
# -----------------------------
frame_metrics = compute_metrics(pred_df["label"], pred_df["fake_prob"], THRESHOLD)
video_metrics = compute_metrics(video_df["label"], video_df["fake_prob"], THRESHOLD)

metrics_df = pd.DataFrame([
    {"Level": "Frame", **frame_metrics},
    {"Level": "Video", **video_metrics},
])

inference_df = pd.DataFrame([{
    "frames_scored": len(pred_df),
    "videos_scored": video_df["video_id"].nunique(),
    "total_seconds": float(elapsed),
    "time_per_frame_ms": float(elapsed / max(len(pred_df), 1) * 1000.0),
    "fps": float(len(pred_df) / max(elapsed, 1e-8)),
    "device": str(device),
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "checkpoint": str(checkpoint),
    "manifest": str(METADATA_CSV),
}])

pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)
video_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)
inference_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(frame_metrics, f, indent=2)

print("\nSaved outputs:")
for f in [
    OUT_DIR / "test_frame_predictions.csv",
    OUT_DIR / "test_video_predictions.csv",
    OUT_DIR / "test_metrics.csv",
    OUT_DIR / "test_inference_time.csv",
    OUT_DIR / "test_metrics.json",
]:
    print("FOUND" if f.exists() else "MISSING", f)

print("\nMetrics:")
display(metrics_df)

print("\nInference time:")
display(inference_df)

Manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest.csv
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_df40_indomain/freqdebias_latest.pt
Script: /home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval

Raw manifest rows: 165663
Raw manifest columns:
['image_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'method', 'domain', 'dataset_partition', 'top_level_folder']

Raw split counts:


,count
split,
external_test,165663



Because this file is already wilddeepfake_test_manifest.csv, using all rows as test.

Rows after normalization: 165663
Missing image files: 0

Counts by label:


,label,count
0,1,107004
1,0,58659



Counts by split and label:


label,0,1
split,,
test,58659,107004


... (4 outputs left out)


Saved outputs:
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval/test_frame_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval/test_video_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval/test_metrics.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval/test_inference_time.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_eval/test_metrics.json

Metrics:


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Real Support,Fake Support,Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate,Mean Fake Probability
0,Frame,0.5,0.646282,0.500582,0.739430,0.649207,0.827957,0.001313,0.002621,0.646180,0.99985,0.785020,58659,107004,93,165570,0.99985,0.00015,0.998899
1,Video,0.5,0.509294,0.500000,0.643831,0.676390,0.000000,0.000000,0.000000,0.509294,1.00000,0.674877,396,411,0,807,1.00000,0.00000,0.998395



Inference time:


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size,checkpoint,manifest
0,165663,807,3826.44022,23.097736,43.294287,cuda,16,256,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...


In [17]:
# ============================================================
# AdaGate DF40 checkpoint -> WildDeepfake evaluation
# Run this entire cell in Jupyter.
# ============================================================

from pathlib import Path
import sys
import shutil
import time
import json

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

# -----------------------------
# Paths / settings
# -----------------------------
PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
REPO_DIR = PROJECT_ROOT / "AdaGate-DF"
SRC_DIR = REPO_DIR / "src"

METADATA_CSV = PROJECT_ROOT / "outputs_wilddeepfake" / "wilddeepfake_test_manifest.csv"

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "outputs_adagate_df40_indomain" / "best_gated_dual_branch_df40.pt",
    PROJECT_ROOT / "outputs_adagate_df40_indomain" / "latest_gated_dual_branch_df40.pt",
    PROJECT_ROOT / "outputs_adagate_df40_smoke" / "best_gated_dual_branch_df40_smoke.pt",
]

OUT_DIR = PROJECT_ROOT / "outputs_adagate_wilddeepfake_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
IMAGE_SIZE = 224
NUM_WORKERS = 0
EVAL_RESOLUTIONS = [224]
THRESHOLD = 0.5
UNCERTAIN_THRESHOLD = 0.65

# -----------------------------
# Prepare AdaGate package import
# -----------------------------
if not REPO_DIR.exists():
    raise FileNotFoundError(f"AdaGate repo not found: {REPO_DIR}")

if not SRC_DIR.exists():
    raise FileNotFoundError(f"AdaGate src folder not found: {SRC_DIR}")

PKG_DIR = SRC_DIR / "deepfake_lowres"
PKG_DIR.mkdir(parents=True, exist_ok=True)

for name in ["data", "models", "training", "results"]:
    src = SRC_DIR / name
    dst = PKG_DIR / name
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        (dst / "__init__.py").touch()

for name in ["metrics.py", "utils.py"]:
    src = SRC_DIR / name
    dst = PKG_DIR / name
    if src.exists():
        shutil.copy2(src, dst)

(PKG_DIR / "__init__.py").touch()

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from deepfake_lowres.data.gated_dataset import GatedMultiResolutionDataset
from deepfake_lowres.models.gated_dual_branch import GatedDualBranchDeepfakeNet

# -----------------------------
# Resolve manifest / checkpoint
# -----------------------------
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"WildDeepfake manifest not found: {METADATA_CSV}")

checkpoint = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if checkpoint is None:
    found = sorted(PROJECT_ROOT.rglob("*adagate*.pt")) + sorted(PROJECT_ROOT.rglob("*gated_dual*.pt"))
    raise FileNotFoundError(
        "Could not find AdaGate checkpoint.\n\nChecked:\n"
        + "\n".join(str(p) for p in CHECKPOINT_CANDIDATES)
        + "\n\nPossible files:\n"
        + "\n".join(str(p) for p in found[:50])
    )

print("Manifest:", METADATA_CSV)
print("Checkpoint:", checkpoint)
print("Output dir:", OUT_DIR)

# -----------------------------
# Load / normalize WildDeepfake manifest
# -----------------------------
raw_df = pd.read_csv(METADATA_CSV)

print("\nRaw manifest rows:", len(raw_df))
print("Raw columns:", list(raw_df.columns))

if len(raw_df) == 0:
    raise ValueError(f"Manifest has zero rows: {METADATA_CSV}")

path_col = None
for c in ["frame_path", "image_path", "path", "filepath", "file_path"]:
    if c in raw_df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"No path column found. Columns: {list(raw_df.columns)}")

if "label" not in raw_df.columns:
    raise ValueError("Manifest must contain label column.")

df = raw_df.copy()
df["image_path"] = df[path_col].astype(str)
df["frame_path"] = df[path_col].astype(str)
df["label"] = df["label"].astype(int)

if "split" in df.columns:
    print("\nRaw split counts:")
    display(df["split"].astype(str).value_counts(dropna=False).to_frame("count"))
    test_mask = df["split"].astype(str).str.lower().str.strip().eq("test")
    if test_mask.sum() > 0:
        df = df[test_mask].reset_index(drop=True)
        print(f"Filtered to split == test: {len(df)} rows")
    else:
        print("No exact split == test rows. Using all rows as test because this is the WildDeepfake test manifest.")
        df["split"] = "test"
else:
    df["split"] = "test"

if "video_id" not in df.columns:
    if "sample_unit_id" in df.columns:
        df["video_id"] = df["sample_unit_id"].astype(str)
    elif "group_id" in df.columns:
        df["video_id"] = df["group_id"].astype(str)
    elif "clip_id" in df.columns:
        df["video_id"] = df["clip_id"].astype(str)
    else:
        df["video_id"] = df["image_path"].map(lambda p: str(Path(p).parent))

if "method" not in df.columns:
    df["method"] = "wilddeepfake"

missing = [p for p in df["image_path"].tolist() if not Path(p).exists()]
print("\nRows after normalization:", len(df))
print("Missing image files:", len(missing))
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} image files. First missing: {missing[0]}")

print("\nCounts by split/label:")
display(pd.crosstab(df["split"], df["label"]))

# -----------------------------
# Metrics
# -----------------------------
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {"Decision Threshold": float(threshold), "Accuracy": float(accuracy_score(y_true, y_pred))}

    if len(np.unique(y_true)) >= 2:
        out["Balanced Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
        out["AUPRC"] = float(average_precision_score(y_true, y_prob))
        out["AUCROC"] = float(roc_auc_score(y_true, y_prob))
    else:
        out["Balanced Accuracy"] = np.nan
        out["AUPRC"] = np.nan
        out["AUCROC"] = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )

    out["Precision (0)"] = float(precision[0])
    out["Recall (0)"] = float(recall[0])
    out["F1 (0)"] = float(f1[0])
    out["Precision (1)"] = float(precision[1])
    out["Recall (1)"] = float(recall[1])
    out["F1 (1)"] = float(f1[1])
    out["Real Support"] = int(support[0])
    out["Fake Support"] = int(support[1])
    out["Predicted Real Count"] = int((y_pred == 0).sum())
    out["Predicted Fake Count"] = int((y_pred == 1).sum())
    out["Fake Detection Rate / Recall(1)"] = float(recall[1])
    out["Missed Fake Rate"] = float(1.0 - recall[1]) if support[1] > 0 else np.nan
    out["Mean Fake Probability"] = float(y_prob.mean())
    return out

# -----------------------------
# Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

ckpt = torch.load(checkpoint, map_location=device)

model = GatedDualBranchDeepfakeNet(branch_channels=(12, 20, 32, 48)).to(device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()

# -----------------------------
# Dataset / loader
# -----------------------------
dataset = GatedMultiResolutionDataset(
    df,
    image_size=IMAGE_SIZE,
    resolutions=EVAL_RESOLUTIONS,
    mode="eval_all",
    cache_features=False,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("Dataset length:", len(dataset))
print("Loader batches:", len(loader))

# -----------------------------
# Predict
# -----------------------------
path_to_video = dict(zip(df["image_path"].astype(str), df["video_id"].astype(str)))
path_to_label = dict(zip(df["image_path"].astype(str), df["label"].astype(int)))

rows = []
start = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="AdaGate WildDeepfake", dynamic_ncols=True):
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)

        pred_out = model.predict(rgb, freq, qf, threshold=UNCERTAIN_THRESHOLD)

        probs = pred_out["probs"].detach().cpu().numpy()
        fake_prob = probs[:, 1]
        pred = pred_out["pred"].detach().cpu().numpy()
        conf = pred_out["conf"].detach().cpu().numpy()
        routes = pred_out["routes"].detach().cpu().numpy()

        labels = batch["label"].detach().cpu().numpy()

        for i in range(len(fake_prob)):
            image_path = str(batch["path"][i])
            rows.append({
                "image_path": image_path,
                "video_id": path_to_video.get(image_path, str(Path(image_path).parent)),
                "label": int(labels[i]),
                "fake_prob": float(fake_prob[i]),
                "pred_label": int(fake_prob[i] >= THRESHOLD),
                "confidence": float(conf[i]),
                "uncertainty": float(1.0 - conf[i]),
                "route": int(routes[i]),
                "method": "wilddeepfake",
            })

elapsed = time.time() - start

pred_df = pd.DataFrame(rows)
if len(pred_df) == 0:
    raise ValueError("Prediction DataFrame is empty.")

# -----------------------------
# Video aggregation / save
# -----------------------------
video_df = pred_df.groupby("video_id").agg(
    label=("label", "first"),
    fake_prob=("fake_prob", "mean"),
    confidence=("confidence", "mean"),
    uncertainty=("uncertainty", "mean"),
    num_frames=("fake_prob", "size"),
).reset_index()
video_df["pred_label"] = (video_df["fake_prob"] >= THRESHOLD).astype(int)

frame_metrics = compute_metrics(pred_df["label"], pred_df["fake_prob"], THRESHOLD)
video_metrics = compute_metrics(video_df["label"], video_df["fake_prob"], THRESHOLD)

metrics_df = pd.DataFrame([
    {"Level": "Frame", **frame_metrics},
    {"Level": "Video", **video_metrics},
])

inference_df = pd.DataFrame([{
    "frames_scored": len(pred_df),
    "videos_scored": video_df["video_id"].nunique(),
    "total_seconds": float(elapsed),
    "time_per_frame_ms": float(elapsed / max(len(pred_df), 1) * 1000.0),
    "fps": float(len(pred_df) / max(elapsed, 1e-8)),
    "device": str(device),
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "checkpoint": str(checkpoint),
    "manifest": str(METADATA_CSV),
}])

pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)
video_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)
inference_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(frame_metrics, f, indent=2)

print("\nSaved outputs:")
for f in [
    OUT_DIR / "test_frame_predictions.csv",
    OUT_DIR / "test_video_predictions.csv",
    OUT_DIR / "test_metrics.csv",
    OUT_DIR / "test_inference_time.csv",
    OUT_DIR / "test_metrics.json",
]:
    print("FOUND" if f.exists() else "MISSING", f)

print("\nMetrics:")
display(metrics_df)

print("\nInference time:")
display(inference_df)

Manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest.csv
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/best_gated_dual_branch_df40.pt
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval

Raw manifest rows: 165663
Raw columns: ['image_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'method', 'domain', 'dataset_partition', 'top_level_folder']

Raw split counts:


,count
split,
external_test,165663


No exact split == test rows. Using all rows as test because this is the WildDeepfake test manifest.

Rows after normalization: 165663
Missing image files: 0

Counts by split/label:


label,0,1
split,,
test,58659,107004



Device: cuda
Dataset length: 165663
Loader batches: 10354


AdaGate WildDeepfake: 100%|██████████| 10354/10354 [55:44<00:00,  3.10it/s] 



Saved outputs:
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval/test_frame_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval/test_video_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval/test_metrics.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval/test_inference_time.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_wilddeepfake_eval/test_metrics.json

Metrics:


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Real Support,Fake Support,Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate,Mean Fake Probability
0,Frame,0.5,0.658300,0.524874,0.701041,0.601764,0.674312,0.067662,0.122984,0.657710,0.982085,0.787815,58659,107004,5886,159777,0.982085,0.017915,0.896172
1,Video,0.5,0.531599,0.522819,0.624428,0.652769,0.909091,0.050505,0.095694,0.521019,0.995134,0.683946,396,411,22,785,0.995134,0.004866,0.882257



Inference time:


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size,checkpoint,manifest
0,165663,807,3344.937905,20.191219,49.52648,cuda,16,224,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...


In [16]:
!pip install opencv-python-headless

  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (61.2 MB)


In [18]:
# ============================================================
# GenD DF40 checkpoint -> WildDeepfake evaluation
# Run this entire cell in Jupyter.
# ============================================================

from pathlib import Path
import importlib.util
import time
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

# -----------------------------
# Paths / settings
# -----------------------------
PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

METADATA_CSV = PROJECT_ROOT / "outputs_wilddeepfake" / "wilddeepfake_test_manifest.csv"
SCRIPT_PATH = PROJECT_ROOT / "gend_reimpl_ffpp.py"

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "outputs_gend_reimpl_df40_indomain" / "best_checkpoint.pt",
    PROJECT_ROOT / "outputs_gend_reimpl_df40_indomain" / "latest_checkpoint.pt",
    PROJECT_ROOT / "outputs_gend_df40_indomain" / "best_checkpoint.pt",
    PROJECT_ROOT / "outputs_gend_df40_indomain" / "latest_checkpoint.pt",
]

OUT_DIR = PROJECT_ROOT / "outputs_gend_wilddeepfake_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
IMAGE_SIZE = 224
NUM_WORKERS = 0
THRESHOLD = 0.5

# -----------------------------
# Resolve inputs
# -----------------------------
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"WildDeepfake manifest not found: {METADATA_CSV}")
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"GenD script not found: {SCRIPT_PATH}")

checkpoint = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if checkpoint is None:
    found = sorted(PROJECT_ROOT.rglob("*gend*.pt")) + sorted(PROJECT_ROOT.rglob("*GenD*.pt"))
    raise FileNotFoundError(
        "Could not find GenD checkpoint.\n\nChecked:\n"
        + "\n".join(str(p) for p in CHECKPOINT_CANDIDATES)
        + "\n\nPossible files:\n"
        + "\n".join(str(p) for p in found[:50])
    )

print("Manifest:", METADATA_CSV)
print("Checkpoint:", checkpoint)
print("Script:", SCRIPT_PATH)
print("Output dir:", OUT_DIR)

# -----------------------------
# Import GenD code
# -----------------------------
def import_module_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

gend_mod = import_module_from_path("gend_reimpl_module", SCRIPT_PATH)

# -----------------------------
# Load / normalize manifest
# -----------------------------
raw_df = pd.read_csv(METADATA_CSV)
print("\nRaw manifest rows:", len(raw_df))
print("Raw columns:", list(raw_df.columns))

if len(raw_df) == 0:
    raise ValueError(f"Manifest has zero rows: {METADATA_CSV}")

path_col = None
for c in ["frame_path", "image_path", "path", "filepath", "file_path"]:
    if c in raw_df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"No path column found. Columns: {list(raw_df.columns)}")

if "label" not in raw_df.columns:
    raise ValueError("Manifest must contain label column.")

df = raw_df.copy()
df["frame_path"] = df[path_col].astype(str)
df["image_path"] = df[path_col].astype(str)
df["label"] = df["label"].astype(int)

if "split" in df.columns:
    print("\nRaw split counts:")
    display(df["split"].astype(str).value_counts(dropna=False).to_frame("count"))
    test_mask = df["split"].astype(str).str.lower().str.strip().eq("test")
    if test_mask.sum() > 0:
        df = df[test_mask].reset_index(drop=True)
        print(f"Filtered to split == test: {len(df)} rows")
    else:
        print("No exact split == test rows. Using all rows as test because this is the WildDeepfake test manifest.")
        df["split"] = "test"
else:
    df["split"] = "test"

if "video_id" not in df.columns:
    if "sample_unit_id" in df.columns:
        df["video_id"] = df["sample_unit_id"].astype(str)
    elif "group_id" in df.columns:
        df["video_id"] = df["group_id"].astype(str)
    elif "clip_id" in df.columns:
        df["video_id"] = df["clip_id"].astype(str)
    else:
        df["video_id"] = df["image_path"].map(lambda p: str(Path(p).parent))

if "method" not in df.columns:
    df["method"] = "wilddeepfake"

missing = [p for p in df["image_path"].tolist() if not Path(p).exists()]
print("\nRows after normalization:", len(df))
print("Missing image files:", len(missing))
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} image files. First missing: {missing[0]}")

print("\nCounts by split/label:")
display(pd.crosstab(df["split"], df["label"]))

# -----------------------------
# Metrics
# -----------------------------
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {"Decision Threshold": float(threshold), "Accuracy": float(accuracy_score(y_true, y_pred))}

    if len(np.unique(y_true)) >= 2:
        out["Balanced Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
        out["AUPRC"] = float(average_precision_score(y_true, y_prob))
        out["AUCROC"] = float(roc_auc_score(y_true, y_prob))
    else:
        out["Balanced Accuracy"] = np.nan
        out["AUPRC"] = np.nan
        out["AUCROC"] = np.nan

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )

    out["Precision (0)"] = float(precision[0])
    out["Recall (0)"] = float(recall[0])
    out["F1 (0)"] = float(f1[0])
    out["Precision (1)"] = float(precision[1])
    out["Recall (1)"] = float(recall[1])
    out["F1 (1)"] = float(f1[1])
    out["Real Support"] = int(support[0])
    out["Fake Support"] = int(support[1])
    out["Predicted Real Count"] = int((y_pred == 0).sum())
    out["Predicted Fake Count"] = int((y_pred == 1).sum())
    out["Fake Detection Rate / Recall(1)"] = float(recall[1])
    out["Missed Fake Rate"] = float(1.0 - recall[1]) if support[1] > 0 else np.nan
    out["Mean Fake Probability"] = float(y_prob.mean())
    return out

# -----------------------------
# Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

ckpt_preview = torch.load(checkpoint, map_location="cpu")
saved_args = ckpt_preview.get("args", {})
if not isinstance(saved_args, dict):
    saved_args = vars(saved_args)

backbone = saved_args.get("backbone", "openai/clip-vit-large-patch14")
train_ln = not bool(saved_args.get("no_ln_tuning", False))
image_size = int(saved_args.get("image_size", IMAGE_SIZE))

if "threshold" in ckpt_preview:
    THRESHOLD = float(ckpt_preview["threshold"])
    print("Using threshold from checkpoint:", THRESHOLD)

print("Backbone:", backbone)
print("Train LN:", train_ln)
print("Image size:", image_size)

model = gend_mod.GenDReimpl(backbone=backbone, train_ln=train_ln).to(device)
gend_mod.load_trainable_checkpoint(model, checkpoint, device)
model.eval()

# -----------------------------
# Dataset / loader
# -----------------------------
dataset = gend_mod.FFPPFrameDataset(
    df,
    image_size=image_size,
    train=False,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("Dataset length:", len(dataset))
print("Loader batches:", len(loader))

if len(dataset) == 0:
    raise ValueError("Dataset length is zero.")

# -----------------------------
# Predict
# -----------------------------
rows = []
start = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="GenD WildDeepfake", dynamic_ncols=True):
        x = batch["image"].to(device, non_blocking=True)

        logits, _ = model(x)
        prob = F.softmax(logits.float(), dim=1)[:, 1].detach().cpu().numpy()

        labels = batch["label"].detach().cpu().numpy()

        for i in range(len(prob)):
            image_path = str(batch["image_path"][i])
            video_id = str(batch["video_id"][i]) if "video_id" in batch else str(Path(image_path).parent)
            rows.append({
                "image_path": image_path,
                "video_id": video_id,
                "label": int(labels[i]),
                "fake_prob": float(prob[i]),
                "pred_label": int(prob[i] >= THRESHOLD),
                "method": "wilddeepfake",
            })

elapsed = time.time() - start

pred_df = pd.DataFrame(rows)
if len(pred_df) == 0:
    raise ValueError("Prediction DataFrame is empty.")

# -----------------------------
# Save
# -----------------------------
video_df = pred_df.groupby("video_id").agg(
    label=("label", "first"),
    fake_prob=("fake_prob", "mean"),
    num_frames=("fake_prob", "size"),
).reset_index()
video_df["pred_label"] = (video_df["fake_prob"] >= THRESHOLD).astype(int)

frame_metrics = compute_metrics(pred_df["label"], pred_df["fake_prob"], THRESHOLD)
video_metrics = compute_metrics(video_df["label"], video_df["fake_prob"], THRESHOLD)

metrics_df = pd.DataFrame([
    {"Level": "Frame", **frame_metrics},
    {"Level": "Video", **video_metrics},
])

inference_df = pd.DataFrame([{
    "frames_scored": len(pred_df),
    "videos_scored": video_df["video_id"].nunique(),
    "total_seconds": float(elapsed),
    "time_per_frame_ms": float(elapsed / max(len(pred_df), 1) * 1000.0),
    "fps": float(len(pred_df) / max(elapsed, 1e-8)),
    "device": str(device),
    "batch_size": BATCH_SIZE,
    "image_size": image_size,
    "checkpoint": str(checkpoint),
    "manifest": str(METADATA_CSV),
}])

pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)
video_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)
inference_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(frame_metrics, f, indent=2)

print("\nSaved outputs:")
for f in [
    OUT_DIR / "test_frame_predictions.csv",
    OUT_DIR / "test_video_predictions.csv",
    OUT_DIR / "test_metrics.csv",
    OUT_DIR / "test_inference_time.csv",
    OUT_DIR / "test_metrics.json",
]:
    print("FOUND" if f.exists() else "MISSING", f)

print("\nMetrics:")
display(metrics_df)

print("\nInference time:")
display(inference_df)

Manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest.csv
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_reimpl_df40_indomain/best_checkpoint.pt
Script: /home/jovyan/Deepfake_Low Resource_Low Resolution/gend_reimpl_ffpp.py
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval

Raw manifest rows: 165663
Raw columns: ['image_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'method', 'domain', 'dataset_partition', 'top_level_folder']

Raw split counts:


,count
split,
external_test,165663


No exact split == test rows. Using all rows as test because this is the WildDeepfake test manifest.

Rows after normalization: 165663
Missing image files: 0

Counts by split/label:


label,0,1
split,,
test,58659,107004



Device: cuda
Using threshold from checkpoint: 0.5
Backbone: openai/clip-vit-large-patch14
Train LN: True
Image size: 224


Loading weights: 100%|██████████| 391/391 [00:02<00:00, 138.37it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight

... (2 outputs left out)


Saved outputs:
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval/test_frame_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval/test_video_predictions.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval/test_metrics.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval/test_inference_time.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_eval/test_metrics.json

Metrics:


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Real Support,Fake Support,Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate,Mean Fake Probability
0,Frame,0.5,0.664524,0.527296,0.698503,0.580226,0.92689,0.057059,0.107500,0.658677,0.997533,0.793441,58659,107004,3611,162052,0.997533,0.002467,0.976086
1,Video,0.5,0.516729,0.507622,0.559201,0.570719,0.87500,0.017677,0.034653,0.513141,0.997567,0.677686,396,411,8,799,0.997567,0.002433,0.989000



Inference time:


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size,checkpoint,manifest
0,165663,807,4244.483224,25.62119,39.030193,cuda,16,224,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...


In [22]:
# ============================================================
# DFD-HR WildDeepfake debug + rerun cell
# Saves full log and prints useful traceback if it fails.
# ============================================================

from pathlib import Path
import subprocess
import sys
import time
import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

METADATA_CSV = PROJECT_ROOT / "outputs_wilddeepfake" / "wilddeepfake_test_manifest.csv"
ADAPTER_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake" / "wilddeepfake_test_manifest_dfdhr_adapter.csv"

DFDHR_REPO = PROJECT_ROOT / "DFD-HR"
DFDHR_RUNNER = DFDHR_REPO / "run_dfd_hr_df40.py"
DFDHR_CKPT = DFDHR_REPO / "ckpt_best.pth"
DETECTOR_YAML = DFDHR_REPO / "training/config/detector/dfd_hr.yaml"
TEST_CONFIG = DFDHR_REPO / "training/config/test_config.yaml"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = OUT_DIR / "dfdhr_wilddeepfake_run.log"

IMAGE_SIZE = 448
FRAMES_PER_VIDEO = 32
BATCH_SIZE = 1

# -----------------------------
# Check paths
# -----------------------------
print("=" * 100)
print("Checking DFD-HR paths")
print("=" * 100)

for label, path in [
    ("WildDeepfake manifest", METADATA_CSV),
    ("DFD-HR repo", DFDHR_REPO),
    ("DFD-HR runner", DFDHR_RUNNER),
    ("DFD-HR checkpoint", DFDHR_CKPT),
    ("Detector YAML", DETECTOR_YAML),
    ("Test config", TEST_CONFIG),
]:
    print(("FOUND   " if path.exists() else "MISSING "), label, ":", path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

# -----------------------------
# Build clean adapter manifest
# -----------------------------
print("\n" + "=" * 100)
print("Building DFD-HR WildDeepfake adapter manifest")
print("=" * 100)

raw_df = pd.read_csv(METADATA_CSV)

print("Raw rows:", len(raw_df))
print("Raw columns:", list(raw_df.columns))

if len(raw_df) == 0:
    raise ValueError(f"Manifest has zero rows: {METADATA_CSV}")

path_col = None
for c in ["frame_path", "image_path", "path", "filepath", "file_path"]:
    if c in raw_df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"No image path column found. Columns: {list(raw_df.columns)}")

if "label" not in raw_df.columns:
    raise ValueError("Manifest must contain label column.")

df = raw_df.copy()
df["frame_path"] = df[path_col].astype(str)
df["split"] = "test"
df["label"] = df["label"].astype(int)
df["label_name"] = df["label"].map({0: "real", 1: "fake"}).fillna(df["label"].astype(str))

if "video_id" not in df.columns:
    if "sample_unit_id" in df.columns:
        df["video_id"] = df["sample_unit_id"].astype(str)
    elif "group_id" in df.columns:
        df["video_id"] = df["group_id"].astype(str)
    elif "clip_id" in df.columns:
        df["video_id"] = df["clip_id"].astype(str)
    else:
        df["video_id"] = df["frame_path"].map(lambda p: str(Path(p).parent))
else:
    df["video_id"] = df["video_id"].astype(str)

if "method" not in df.columns:
    df["method"] = "wilddeepfake"
else:
    df["method"] = df["method"].astype(str)

# IMPORTANT: do not include image_path here, because some loaders rename frame_path -> image_path.
adapter_cols = [
    "frame_path",
    "split",
    "label",
    "label_name",
    "video_id",
    "method",
]

missing_images = [p for p in df["frame_path"].tolist() if not Path(p).exists()]
print("Rows after normalization:", len(df))
print("Missing image files:", len(missing_images))

if missing_images:
    print("First missing:", missing_images[0])
    raise FileNotFoundError(f"Missing {len(missing_images)} image files.")

df[adapter_cols].to_csv(ADAPTER_MANIFEST, index=False)

print("Saved adapter manifest:", ADAPTER_MANIFEST)
print("\nCounts by split/label:")
display(pd.crosstab(df["split"], df["label"]))

print("\nFirst rows:")
display(df[adapter_cols].head())

# -----------------------------
# Show runner help briefly
# -----------------------------
print("\n" + "=" * 100)
print("Checking DFD-HR runner help")
print("=" * 100)

help_result = subprocess.run(
    [sys.executable, str(DFDHR_RUNNER), "--help"],
    cwd=str(DFDHR_REPO),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(help_result.stdout[:3000])

# -----------------------------
# Run DFD-HR
# -----------------------------
print("\n" + "=" * 100)
print("Running DFD-HR WildDeepfake evaluation")
print("=" * 100)

cmd = [
    sys.executable,
    str(DFDHR_RUNNER),
    "--repo_root",
    str(DFDHR_REPO),
    "--metadata_csv",
    str(ADAPTER_MANIFEST),
    "--ckpt_path",
    str(DFDHR_CKPT),
    "--detector_yaml",
    str(DETECTOR_YAML),
    "--test_config",
    str(TEST_CONFIG),
    "--out_dir",
    str(OUT_DIR),
    "--split",
    "test",
    "--image_size",
    str(IMAGE_SIZE),
    "--frames_per_video",
    str(FRAMES_PER_VIDEO),
    "--batch_size",
    str(BATCH_SIZE),
    "--allow_non_strict",
]

print("Command:")
print(" ".join(cmd))
print("Log file:")
print(LOG_PATH)

start = time.time()

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd=str(DFDHR_REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    # Print only occasional lines live, but write everything to log.
    live_count = 0
    for line in proc.stdout:
        log.write(line)
        live_count += 1

        # Print informative lines and every 100th line.
        low = line.lower()
        if (
            "error" in low
            or "traceback" in low
            or "runtimeerror" in low
            or "loading" in low
            or "predict" in low
            or "test" in low
            or live_count % 100 == 0
        ):
            print(line.rstrip())

    proc.wait()

elapsed = time.time() - start

print("\nReturn code:", proc.returncode)
print("Elapsed seconds:", elapsed)
print("Full log:", LOG_PATH)

if proc.returncode != 0:
    print("\n" + "=" * 100)
    print("DFD-HR failed. Last 120 log lines:")
    print("=" * 100)

    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-120:]:
        print(line)

    raise RuntimeError(f"DFD-HR WildDeepfake evaluation failed with return code {proc.returncode}")

# -----------------------------
# Show outputs
# -----------------------------
print("\n" + "=" * 100)
print("DFD-HR finished. Output files:")
print("=" * 100)

for name in [
    "test_metrics.csv",
    "metrics.csv",
    "frame_predictions.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_inference_time.csv",
    "inference_time.csv",
]:
    p = OUT_DIR / name
    print(("FOUND   " if p.exists() else "MISSING "), p)

for name in ["test_metrics.csv", "metrics.csv"]:
    p = OUT_DIR / name
    if p.exists():
        print("\nMetrics:", p)
        display(pd.read_csv(p))

for name in ["test_inference_time.csv", "inference_time.csv"]:
    p = OUT_DIR / name
    if p.exists():
        print("\nInference:", p)
        display(pd.read_csv(p))

Checking DFD-HR paths
FOUND    WildDeepfake manifest : /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest.csv
FOUND    DFD-HR repo : /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR
FOUND    DFD-HR runner : /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/run_dfd_hr_df40.py
FOUND    DFD-HR checkpoint : /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/ckpt_best.pth
FOUND    Detector YAML : /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/config/detector/dfd_hr.yaml
FOUND    Test config : /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/config/test_config.yaml

Building DFD-HR WildDeepfake adapter manifest
Raw rows: 165663
Raw columns: ['image_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'method', 'domain', 'dataset_partition', 'top_level_folder']
Rows after normalization: 165663
Missing image files: 0
Saved adapter manifest: /home/jovyan/Deepfake_L

label,0,1
split,,
test,58659,107004



First rows:


,frame_path,split,label,label_name,video_id,method
0,/home/jovyan/Deepfake_Low Resource_Low Resolut...,test,0,real,real::104/104/real/10,wilddeepfake
1,/home/jovyan/Deepfake_Low Resource_Low Resolut...,test,0,real,real::104/104/real/10,wilddeepfake
2,/home/jovyan/Deepfake_Low Resource_Low Resolut...,test,0,real,real::104/104/real/10,wilddeepfake
3,/home/jovyan/Deepfake_Low Resource_Low Resolut...,test,0,real,real::104/104/real/10,wilddeepfake
4,/home/jovyan/Deepfake_Low Resource_Low Resolut...,test,0,real,real::104/104/real/10,wilddeepfake



Checking DFD-HR runner help
usage: Run DFD-HR on DF40 manifest [-h] [--repo_root REPO_ROOT] --metadata_csv
                                   METADATA_CSV --ckpt_path CKPT_PATH
                                   [--detector_yaml DETECTOR_YAML]
                                   [--test_config TEST_CONFIG]
                                   [--out_dir OUT_DIR]
                                   [--split {train,val,test}]
                                   [--image_size IMAGE_SIZE]
                                   [--frames_per_video FRAMES_PER_VIDEO]
                                   [--batch_size BATCH_SIZE]
                                   [--num_workers NUM_WORKERS]
                                   [--limit_videos LIMIT_VIDEOS]
                                   [--path_col PATH_COL]
                                   [--label_col LABEL_COL]
                                   [--split_col SPLIT_COL]
                                   [--group_col GROUP_COL] [--verify_images]


,Level,Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),Precision (0),Recall (0),F1 (0),Precision (1),Recall (1),F1 (1),Uncertain Rate (%)
0,Frame,0.726088,0.871050,0.857692,0.221843,0.827397,0.559107,0.667295,0.675740,0.887356,0.767223,19.009033
1,Video,0.737299,0.904563,0.896274,0.179671,0.889831,0.530303,0.664557,0.674256,0.936740,0.784114,23.420074



Inference: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_eval/inference_time.csv


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size
0,25793,807,12093.521676,468.868363,2.132795,cuda,1,448


In [21]:
# !pip install numpy pandas scikit-learn transformers torchvision
!pip install einops

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)


In [28]:
# ============================================================
# STEP 2A: Robust WildDeepfake train download using Hugging Face API
# Downloads missing:
#   deepfake_in_the_wild/real_train/*.tar.gz
#   deepfake_in_the_wild/fake_train/*.tar.gz
# ============================================================

from pathlib import Path
import os
import sys
import shutil
import subprocess
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
WILD_ROOT = PROJECT_ROOT / "WildDeepfake"
DATASET_ROOT = WILD_ROOT / "deepfake_in_the_wild"

WILD_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Installing / upgrading Hugging Face tools")
print("=" * 100)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "huggingface_hub", "hf_xet"],
    check=True,
)

from huggingface_hub import HfApi, hf_hub_download

print("\n" + "=" * 100)
print("Disk space check")
print("=" * 100)

usage = shutil.disk_usage(PROJECT_ROOT)
free_gb = usage.free / 1024**3
print("Total GB:", round(usage.total / 1024**3, 2))
print("Used  GB:", round(usage.used / 1024**3, 2))
print("Free  GB:", round(free_gb, 2))

if free_gb < 80:
    print("\nWARNING: You have less than 80 GB free.")
    print("WildDeepfake train archives are large. Download may fail if disk space runs out.")

print("\n" + "=" * 100)
print("Remote Hugging Face file listing")
print("=" * 100)

repo_id = "xingjunm/WildDeepfake"
api = HfApi()

try:
    remote_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
except Exception as e:
    print("Could not list repo files.")
    print("Error:", repr(e))
    print("\nIf this is an authentication/rate-limit issue, run:")
    print("from huggingface_hub import login")
    print("login()")
    raise

train_files = [
    f for f in remote_files
    if f.startswith("deepfake_in_the_wild/real_train/") or f.startswith("deepfake_in_the_wild/fake_train/")
]
train_files = [f for f in train_files if f.endswith(".tar.gz")]

real_train_files = [f for f in train_files if "/real_train/" in f]
fake_train_files = [f for f in train_files if "/fake_train/" in f]

print("Remote real_train archives:", len(real_train_files))
print("Remote fake_train archives:", len(fake_train_files))

if len(real_train_files) == 0 or len(fake_train_files) == 0:
    print("\nFirst 100 remote files:")
    for f in remote_files[:100]:
        print(f)
    raise RuntimeError("Could not find real_train/fake_train tar.gz files in remote listing.")

print("\nFirst 10 real_train files:")
for f in real_train_files[:10]:
    print(" ", f)

print("\nFirst 10 fake_train files:")
for f in fake_train_files[:10]:
    print(" ", f)

print("\n" + "=" * 100)
print("Local archive count before download")
print("=" * 100)

for folder in ["real_train", "fake_train", "real_test", "fake_test"]:
    folder_path = DATASET_ROOT / folder
    archives = sorted(folder_path.glob("*.tar.gz")) if folder_path.exists() else []
    print(f"{folder:12s} exists={folder_path.exists()} archives={len(archives)}")

print("\n" + "=" * 100)
print("Downloading missing train archives one by one")
print("=" * 100)

all_targets = real_train_files + fake_train_files

failed = []
downloaded = 0
already_present = 0

for idx, remote_path in enumerate(all_targets, start=1):
    local_path = WILD_ROOT / remote_path

    if local_path.exists() and local_path.stat().st_size > 0:
        already_present += 1
        if idx % 25 == 0 or idx == len(all_targets):
            print(f"[{idx}/{len(all_targets)}] already present count={already_present}, downloaded={downloaded}")
        continue

    print("\n" + "-" * 100)
    print(f"[{idx}/{len(all_targets)}] Downloading:")
    print(remote_path)

    try:
        t0 = time.time()

        downloaded_path = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=remote_path,
            local_dir=str(WILD_ROOT),
            local_dir_use_symlinks=False,
            resume_download=True,
        )

        elapsed = time.time() - t0
        downloaded += 1

        p = Path(downloaded_path)
        print("Saved:", p)
        print("Size MB:", round(p.stat().st_size / 1024**2, 2))
        print("Seconds:", round(elapsed, 2))

    except Exception as e:
        print("FAILED:", remote_path)
        print("Error type:", type(e).__name__)
        print("Error:", repr(e))
        failed.append((remote_path, repr(e)))

        # Stop immediately so we can inspect the real error.
        break

print("\n" + "=" * 100)
print("Local archive count after download")
print("=" * 100)

for folder in ["real_train", "fake_train", "real_test", "fake_test"]:
    folder_path = DATASET_ROOT / folder
    archives = sorted(folder_path.glob("*.tar.gz")) if folder_path.exists() else []
    total_mb = sum(p.stat().st_size for p in archives) / 1024**2 if archives else 0
    print(f"{folder:12s} exists={folder_path.exists()} archives={len(archives)} size_mb={round(total_mb, 2)}")

if failed:
    print("\n" + "=" * 100)
    print("Download stopped because one file failed")
    print("=" * 100)
    for remote_path, err in failed:
        print(remote_path)
        print(err)
    raise RuntimeError("At least one training archive failed to download. See the error above.")

print("\nDONE. real_train and fake_train should now have archives.")

Installing / upgrading Hugging Face tools

Disk space check
Total GB: 199.94
Used  GB: 108.85
Free  GB: 91.08

Remote Hugging Face file listing


Remote real_train archives: 371
Remote fake_train archives: 592

First 10 real_train files:
  deepfake_in_the_wild/real_train/1.tar.gz
  deepfake_in_the_wild/real_train/101.tar.gz
  deepfake_in_the_wild/real_train/102.tar.gz
  deepfake_in_the_wild/real_train/104.tar.gz
  deepfake_in_the_wild/real_train/105.tar.gz
  deepfake_in_the_wild/real_train/107.tar.gz
  deepfake_in_the_wild/real_train/108.tar.gz
  deepfake_in_the_wild/real_train/109.tar.gz
  deepfake_in_the_wild/real_train/11.tar.gz
  deepfake_in_the_wild/real_train/110.tar.gz

First 10 fake_train files:
  deepfake_in_the_wild/fake_train/1.tar.gz
  deepfake_in_the_wild/fake_train/10.tar.gz
  deepfake_in_the_wild/fake_train/100.tar.gz
  deepfake_in_the_wild/fake_train/101.tar.gz
  deepfake_in_the_wild/fake_train/102.tar.gz
  deepfake_in_the_wild/fake_train/103.tar.gz
  deepfake_in_the_wild/fake_train/104.tar.gz
  deepfake_in_the_wild/fake_train/105.tar.gz
  deepfake_in_the_wild/fake_train/106.tar.gz
  deepfake_in_the_wild/fake_tra

In [30]:
# ============================================================
# STEP 3 FIXED: Robust WildDeepfake extraction + canonical manifest
# Handles .tar.gz files that are actually plain .tar archives.
# ============================================================

from pathlib import Path
import tarfile
import shutil
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

WILD_ROOT = PROJECT_ROOT / "WildDeepfake"
DATASET_ROOT = WILD_ROOT / "deepfake_in_the_wild"
EXTRACT_ROOT = DATASET_ROOT / "extracted"

OUT_DIR = PROJECT_ROOT / "outputs_wilddeepfake"
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

VAL_FRACTION = 0.10
SEED = 42

folders = [
    ("real_train", 0, "train"),
    ("fake_train", 1, "train"),
    ("real_test", 0, "test"),
    ("fake_test", 1, "test"),
]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def print_header(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def safe_extract_tar(tar_path, dest):
    """
    Extract tar safely and support both gzipped and non-gzipped tar files.
    tarfile mode r:* auto-detects compression.
    """
    dest = Path(dest).resolve()

    with tarfile.open(tar_path, mode="r:*") as tf:
        members = tf.getmembers()

        for member in members:
            member_path = (dest / member.name).resolve()
            if not str(member_path).startswith(str(dest)):
                raise RuntimeError(f"Unsafe path in archive {tar_path}: {member.name}")

        tf.extractall(dest)


def archive_magic(path, n=8):
    with open(path, "rb") as f:
        return f.read(n)


# ------------------------------------------------------------
# Disk and archive check
# ------------------------------------------------------------

print_header("Disk space check")

usage = shutil.disk_usage(PROJECT_ROOT)
print("Total GB:", round(usage.total / 1024**3, 2))
print("Used  GB:", round(usage.used / 1024**3, 2))
print("Free  GB:", round(usage.free / 1024**3, 2))

print_header("Archive count and magic check")

for folder, label, split in folders:
    archive_dir = DATASET_ROOT / folder
    archives = sorted(archive_dir.glob("*.tar.gz")) if archive_dir.exists() else []

    print(f"{folder:12s} exists={archive_dir.exists()} archives={len(archives)} path={archive_dir}")

    if len(archives) == 0:
        raise FileNotFoundError(f"No archives found for {folder}: {archive_dir}")

    first = archives[0]
    magic = archive_magic(first)
    print("  first archive:", first.name, "magic:", magic)

print_header("Extracting archives with auto-detected tar mode")

failed = []

for folder, label, split in folders:
    archive_dir = DATASET_ROOT / folder
    target_dir = EXTRACT_ROOT / folder
    target_dir.mkdir(parents=True, exist_ok=True)

    archives = sorted(archive_dir.glob("*.tar.gz"))

    for tar_path in tqdm(archives, desc=f"Extract {folder}", dynamic_ncols=True):
        archive_name = tar_path.name.replace(".tar.gz", "")
        dest = target_dir / archive_name
        marker = dest / ".extracted_done"

        if marker.exists():
            continue

        # If a previous failed attempt made a partial folder, remove it.
        if dest.exists():
            shutil.rmtree(dest)

        dest.mkdir(parents=True, exist_ok=True)

        try:
            safe_extract_tar(tar_path, dest)
            marker.write_text("done")
        except Exception as e:
            failed.append({
                "folder": folder,
                "archive": str(tar_path),
                "destination": str(dest),
                "error": repr(e),
                "magic": str(archive_magic(tar_path)),
            })

            if dest.exists():
                shutil.rmtree(dest)

            print("\nFAILED extracting:", tar_path)
            print("Magic bytes:", archive_magic(tar_path))
            print("Error:", repr(e))
            break

    if failed:
        break

if failed:
    failed_df = pd.DataFrame(failed)
    failed_path = OUT_DIR / "ERROR_wilddeepfake_failed_extractions.csv"
    failed_df.to_csv(failed_path, index=False)
    display(failed_df)
    raise RuntimeError(f"Extraction failed. Saved details to: {failed_path}")

print("\nExtraction complete.")

# ------------------------------------------------------------
# Build manifest
# ------------------------------------------------------------

print_header("Building WildDeepfake canonical manifest")

rows = []

for folder, label, split in folders:
    root = EXTRACT_ROOT / folder
    label_name = "real" if label == 0 else "fake"

    image_paths = sorted([
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and p.name != ".extracted_done"
    ])

    print(f"{folder:12s} images={len(image_paths)} root={root}")

    if len(image_paths) == 0:
        raise RuntimeError(f"No images found after extraction for {folder}: {root}")

    for p in tqdm(image_paths, desc=f"Manifest {folder}", dynamic_ncols=True):
        rel_to_folder = p.relative_to(root)
        parts = rel_to_folder.parts

        archive_id = parts[0] if len(parts) >= 1 else p.parent.name

        if len(parts) >= 3:
            inner_unit = "/".join(parts[1:-1])
        elif len(parts) == 2:
            inner_unit = archive_id
        else:
            inner_unit = str(p.parent.relative_to(root))

        sample_unit_id = f"wilddeepfake::{folder}::{archive_id}::{inner_unit}"

        rows.append({
            "frame_path": str(p),
            "image_path": str(p),
            "label": int(label),
            "label_name": label_name,
            "split": split,
            "sample_unit_id": sample_unit_id,
            "group_id": sample_unit_id,
            "video_id": sample_unit_id,
            "clip_id": sample_unit_id,
            "method": "wilddeepfake",
            "domain": "wilddeepfake",
            "dataset_partition": f"wilddeepfake_{folder}",
            "top_level_folder": folder,
        })

df = pd.DataFrame(rows)

print("\nRaw manifest rows:", len(df))

print("\nCounts before validation split:")
display(pd.crosstab(df["split"], df["label"]))

print("\nUnique sample units before validation split:")
display(df.groupby(["split", "label"])["sample_unit_id"].nunique().unstack(fill_value=0))

# ------------------------------------------------------------
# Create validation split from train groups only
# ------------------------------------------------------------

print_header("Creating validation split from training sample units only")

rng = np.random.default_rng(SEED)

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

val_units = []

for label, g in train_df.groupby("label"):
    units = np.array(sorted(g["sample_unit_id"].unique()))
    rng.shuffle(units)

    n_val = max(1, int(round(VAL_FRACTION * len(units))))
    selected = units[:n_val].tolist()
    val_units.extend(selected)

    print(f"label={label} train_units={len(units)} val_units={len(selected)}")

val_units = set(val_units)

train_df.loc[train_df["sample_unit_id"].isin(val_units), "split"] = "val"

df_final = pd.concat([train_df, test_df], ignore_index=True)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

print_header("Safety checks")

required_splits = ["train", "val", "test"]
observed_splits = sorted(df_final["split"].unique().tolist())
print("Observed splits:", observed_splits)

missing_splits = [s for s in required_splits if s not in observed_splits]
if missing_splits:
    raise RuntimeError(f"Missing required split(s): {missing_splits}")

print("\nFinal counts by split and label:")
split_counts = pd.crosstab(df_final["split"], df_final["label"])
display(split_counts)

for split_name in required_splits:
    labels = set(df_final[df_final["split"] == split_name]["label"].unique().tolist())
    if labels != {0, 1}:
        raise RuntimeError(f"Split {split_name} does not contain both classes. Labels found: {labels}")

print("\nUnique sample units by split and label:")
unit_counts = df_final.groupby(["split", "label"])["sample_unit_id"].nunique().unstack(fill_value=0)
display(unit_counts)

duplicate_path_count = int(df_final["image_path"].duplicated().sum())
print("\nDuplicate image path count:", duplicate_path_count)

if duplicate_path_count > 0:
    dup_path = OUT_DIR / "ERROR_wilddeepfake_duplicate_paths.csv"
    df_final[df_final["image_path"].duplicated(keep=False)].to_csv(dup_path, index=False)
    raise RuntimeError(f"Duplicate image paths found. Saved to: {dup_path}")

path_leak = df_final.groupby("image_path")["split"].nunique()
path_leak_count = int((path_leak > 1).sum())
print("Image path split leakage count:", path_leak_count)

if path_leak_count > 0:
    leak_path = OUT_DIR / "ERROR_wilddeepfake_path_leakage.csv"
    leaked_paths = path_leak[path_leak > 1].index
    df_final[df_final["image_path"].isin(leaked_paths)].to_csv(leak_path, index=False)
    raise RuntimeError(f"Image path leakage found. Saved to: {leak_path}")

group_leak = df_final.groupby("sample_unit_id")["split"].nunique()
group_leak_count = int((group_leak > 1).sum())
print("Group/sample unit split leakage count:", group_leak_count)

if group_leak_count > 0:
    leak_path = OUT_DIR / "ERROR_wilddeepfake_group_leakage.csv"
    leaked_units = group_leak[group_leak > 1].index
    df_final[df_final["sample_unit_id"].isin(leaked_units)].to_csv(leak_path, index=False)
    raise RuntimeError(f"Group leakage found. Saved to: {leak_path}")

missing_images = [p for p in df_final["image_path"].tolist() if not Path(p).exists()]
print("Missing image files:", len(missing_images))

if missing_images:
    missing_path = OUT_DIR / "ERROR_wilddeepfake_missing_images.csv"
    df_final[df_final["image_path"].isin(missing_images)].to_csv(missing_path, index=False)
    raise FileNotFoundError(f"Missing image files. First missing: {missing_images[0]}")

# ------------------------------------------------------------
# Save manifests
# ------------------------------------------------------------

canonical_path = OUT_DIR / "wilddeepfake_manifest_canonical.csv"
train_path = OUT_DIR / "wilddeepfake_train_manifest.csv"
val_path = OUT_DIR / "wilddeepfake_val_manifest.csv"
test_path = OUT_DIR / "wilddeepfake_test_manifest_canonical.csv"
audit_path = OUT_DIR / "wilddeepfake_manifest_audit.csv"

df_final.to_csv(canonical_path, index=False)
df_final[df_final["split"] == "train"].to_csv(train_path, index=False)
df_final[df_final["split"] == "val"].to_csv(val_path, index=False)
df_final[df_final["split"] == "test"].to_csv(test_path, index=False)

audit_df = pd.DataFrame([{
    "canonical_manifest": str(canonical_path),
    "total_rows": len(df_final),
    "train_rows": int((df_final["split"] == "train").sum()),
    "val_rows": int((df_final["split"] == "val").sum()),
    "test_rows": int((df_final["split"] == "test").sum()),
    "real_rows": int((df_final["label"] == 0).sum()),
    "fake_rows": int((df_final["label"] == 1).sum()),
    "duplicate_path_count": duplicate_path_count,
    "path_leakage_count": path_leak_count,
    "group_leakage_count": group_leak_count,
    "missing_image_count": len(missing_images),
    "val_fraction": VAL_FRACTION,
    "seed": SEED,
}])
audit_df.to_csv(audit_path, index=False)

print_header("Saved WildDeepfake canonical manifests")

for p in [canonical_path, train_path, val_path, test_path, audit_path]:
    print("FOUND" if p.exists() else "MISSING", p)

print("\nCanonical manifest is ready for training:")
print(canonical_path)


Disk space check
Total GB: 199.94
Used  GB: 167.42
Free  GB: 32.51

Archive count and magic check
real_train   exists=True archives=371 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_train
  first archive: 1.tar.gz magic: b'./1/\x00\x00\x00\x00'
fake_train   exists=True archives=592 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_train
  first archive: 1.tar.gz magic: b'./1/\x00\x00\x00\x00'
real_test    exists=True archives=42 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/real_test
  first archive: 104.tar.gz magic: b'./104/\x00\x00'
fake_test    exists=True archives=115 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_test
  first archive: 1.tar.gz magic: b'./1/\x00\x00\x00\x00'

Extracting archives with auto-detected tar mode


Extract fake_train:  27%|██▋       | 159/592 [29:45<1:21:03, 11.23s/it]


FAILED extracting: /home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/fake_train/242.tar.gz
Magic bytes: b'./242/\x00\x00'
Error: OSError(28, 'No space left on device')


,folder,archive,destination,error,magic
0,fake_train,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...,"OSError(28, 'No space left on device')",b'./242/\x00\x00'


RuntimeError: Extraction failed. Saved details to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/ERROR_wilddeepfake_failed_extractions.csv

In [31]:
# ============================================================
# Resume WildDeepfake extraction after "No space left on device"
# Deletes only successfully extracted archives to free space.
# Continues extraction and deletes each archive after success.
# ============================================================

from pathlib import Path
import tarfile
import shutil
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

WILD_ROOT = PROJECT_ROOT / "WildDeepfake"
DATASET_ROOT = WILD_ROOT / "deepfake_in_the_wild"
EXTRACT_ROOT = DATASET_ROOT / "extracted"
OUT_DIR = PROJECT_ROOT / "outputs_wilddeepfake"
OUT_DIR.mkdir(parents=True, exist_ok=True)

folders = [
    "real_train",
    "fake_train",
    "real_test",
    "fake_test",
]

def gb(n):
    return round(n / 1024**3, 2)

def disk_report(title):
    usage = shutil.disk_usage(PROJECT_ROOT)
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    print("Total GB:", gb(usage.total))
    print("Used  GB:", gb(usage.used))
    print("Free  GB:", gb(usage.free))

def dir_size(path):
    path = Path(path)
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except Exception:
                pass
    return total

def safe_extract_tar(tar_path, dest):
    """
    Auto-detects plain tar vs gzip tar.
    Also blocks unsafe paths inside archives.
    """
    dest = Path(dest).resolve()

    with tarfile.open(tar_path, mode="r:*") as tf:
        members = tf.getmembers()

        for member in members:
            member_path = (dest / member.name).resolve()
            if not str(member_path).startswith(str(dest)):
                raise RuntimeError(f"Unsafe path in archive {tar_path}: {member.name}")

        tf.extractall(dest)

def count_status():
    rows = []
    for folder in folders:
        archive_dir = DATASET_ROOT / folder
        extract_dir = EXTRACT_ROOT / folder

        archives = sorted(archive_dir.glob("*.tar.gz")) if archive_dir.exists() else []
        markers = sorted(extract_dir.glob("*/*.extracted_done")) if extract_dir.exists() else []

        rows.append({
            "folder": folder,
            "remaining_archives": len(archives),
            "successfully_extracted_archives": len(markers),
            "archive_dir_gb": gb(dir_size(archive_dir)),
            "extract_dir_gb": gb(dir_size(extract_dir)),
            "archive_dir": str(archive_dir),
            "extract_dir": str(extract_dir),
        })

    return pd.DataFrame(rows)

disk_report("Disk before cleanup")
status_before = count_status()
display(status_before)

# ------------------------------------------------------------
# 1. Remove failed partial extraction folder
# ------------------------------------------------------------

failed_partial = EXTRACT_ROOT / "fake_train" / "242"
if failed_partial.exists() and not (failed_partial / ".extracted_done").exists():
    print("\nRemoving failed partial folder:", failed_partial)
    shutil.rmtree(failed_partial)
else:
    print("\nNo failed partial fake_train/242 folder to remove, or it already completed.")

# ------------------------------------------------------------
# 2. Delete archives that already have .extracted_done markers
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("Deleting archives that were already successfully extracted")
print("=" * 100)

deleted_count = 0
deleted_bytes = 0

for folder in folders:
    archive_dir = DATASET_ROOT / folder
    extract_dir = EXTRACT_ROOT / folder

    if not archive_dir.exists() or not extract_dir.exists():
        continue

    for marker in sorted(extract_dir.glob("*/*.extracted_done")):
        archive_id = marker.parent.name
        archive_path = archive_dir / f"{archive_id}.tar.gz"

        if archive_path.exists():
            size = archive_path.stat().st_size
            archive_path.unlink()
            deleted_count += 1
            deleted_bytes += size

print("Deleted archives:", deleted_count)
print("Freed GB:", gb(deleted_bytes))

disk_report("Disk after deleting already extracted archives")
status_after_cleanup = count_status()
display(status_after_cleanup)

# ------------------------------------------------------------
# 3. Resume extraction.
# Delete each archive immediately after successful extraction.
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("Resuming extraction with delete-after-success")
print("=" * 100)

failed = []

for folder in folders:
    archive_dir = DATASET_ROOT / folder
    extract_dir = EXTRACT_ROOT / folder
    extract_dir.mkdir(parents=True, exist_ok=True)

    archives = sorted(archive_dir.glob("*.tar.gz")) if archive_dir.exists() else []

    print(f"\nFolder: {folder}")
    print("Remaining archives to process:", len(archives))

    for tar_path in tqdm(archives, desc=f"Extract {folder}", dynamic_ncols=True):
        archive_id = tar_path.name.replace(".tar.gz", "")
        dest = extract_dir / archive_id
        marker = dest / ".extracted_done"

        # If already completed, delete archive and skip.
        if marker.exists():
            try:
                size = tar_path.stat().st_size
                tar_path.unlink()
                deleted_count += 1
                deleted_bytes += size
            except Exception:
                pass
            continue

        # Remove partial folder if present.
        if dest.exists():
            shutil.rmtree(dest)

        dest.mkdir(parents=True, exist_ok=True)

        try:
            safe_extract_tar(tar_path, dest)
            marker.write_text("done")

            # Critical space saver: delete archive after successful extraction.
            size = tar_path.stat().st_size
            tar_path.unlink()
            deleted_count += 1
            deleted_bytes += size

        except Exception as e:
            if dest.exists():
                shutil.rmtree(dest)

            failed.append({
                "folder": folder,
                "archive": str(tar_path),
                "destination": str(dest),
                "error": repr(e),
            })

            failed_path = OUT_DIR / "ERROR_wilddeepfake_failed_extractions_resume.csv"
            pd.DataFrame(failed).to_csv(failed_path, index=False)

            print("\nFAILED extracting:", tar_path)
            print("Error:", repr(e))
            print("Saved failure details:", failed_path)
            disk_report("Disk at failure")
            raise RuntimeError(f"Extraction failed again. Saved details to: {failed_path}")

disk_report("Disk after resumed extraction")

status_final = count_status()
display(status_final)

print("\nDeleted total archives:", deleted_count)
print("Freed total GB:", gb(deleted_bytes))

print("\nDONE. If remaining_archives are all 0, extraction is complete.")


Disk before cleanup
Total GB: 199.94
Used  GB: 164.17
Free  GB: 35.76


,folder,remaining_archives,successfully_extracted_archives,archive_dir_gb,extract_dir_gb,archive_dir,extract_dir
0,real_train,371,371,21.42,21.15,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
1,fake_train,592,159,37.13,9.78,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
2,real_test,42,0,3.24,3.19,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
3,fake_test,115,0,6.01,5.94,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...



No failed partial fake_train/242 folder to remove, or it already completed.

Deleting archives that were already successfully extracted
Deleted archives: 530
Freed GB: 31.33

Disk after deleting already extracted archives
Total GB: 199.94
Used  GB: 132.84
Free  GB: 67.09


,folder,remaining_archives,successfully_extracted_archives,archive_dir_gb,extract_dir_gb,archive_dir,extract_dir
0,real_train,0,371,0.00,21.15,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
1,fake_train,433,159,27.22,9.78,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
2,real_test,42,0,3.24,3.19,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
3,fake_test,115,0,6.01,5.94,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...



Resuming extraction with delete-after-success

Folder: real_train
Remaining archives to process: 0


Extract real_train: 0it [00:00, ?it/s]


... (5 outputs left out)

Extract fake_test: 100%|██████████| 115/115 [14:49<00:00,  7.74s/it]



Disk after resumed extraction
Total GB: 199.94
Used  GB: 116.83
Free  GB: 83.11


,folder,remaining_archives,successfully_extracted_archives,archive_dir_gb,extract_dir_gb,archive_dir,extract_dir
0,real_train,0,371,0.0,21.15,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
1,fake_train,0,592,0.0,36.67,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
2,real_test,0,42,0.0,3.19,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...
3,fake_test,0,115,0.0,5.94,/home/jovyan/Deepfake_Low Resource_Low Resolut...,/home/jovyan/Deepfake_Low Resource_Low Resolut...



Deleted total archives: 1120
Freed total GB: 67.8

DONE. If remaining_archives are all 0, extraction is complete.


In [32]:
# ============================================================
# Build WildDeepfake canonical manifest from extracted folders
# Creates train/val/test manifest for model training.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

EXTRACT_ROOT = PROJECT_ROOT / "WildDeepfake/deepfake_in_the_wild/extracted"
OUT_DIR = PROJECT_ROOT / "outputs_wilddeepfake"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

VAL_FRACTION = 0.10
SEED = 42

folders = [
    ("real_train", 0, "train"),
    ("fake_train", 1, "train"),
    ("real_test", 0, "test"),
    ("fake_test", 1, "test"),
]

print("=" * 100)
print("Checking extracted folders")
print("=" * 100)

for folder, label, split in folders:
    root = EXTRACT_ROOT / folder
    if not root.exists():
        raise FileNotFoundError(f"Missing extracted folder: {root}")

    markers = list(root.glob("*/*.extracted_done"))
    print(f"{folder:12s} exists=True extracted_archives={len(markers)} path={root}")

print("\n" + "=" * 100)
print("Building manifest rows")
print("=" * 100)

rows = []

for folder, label, split in folders:
    root = EXTRACT_ROOT / folder
    label_name = "real" if label == 0 else "fake"

    image_paths = sorted([
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and p.name != ".extracted_done"
    ])

    print(f"{folder:12s} images={len(image_paths)}")

    if len(image_paths) == 0:
        raise RuntimeError(f"No images found in {root}")

    for p in tqdm(image_paths, desc=f"Manifest {folder}", dynamic_ncols=True):
        rel_to_folder = p.relative_to(root)
        parts = rel_to_folder.parts

        archive_id = parts[0] if len(parts) >= 1 else p.parent.name

        if len(parts) >= 3:
            inner_unit = "/".join(parts[1:-1])
        elif len(parts) == 2:
            inner_unit = archive_id
        else:
            inner_unit = str(p.parent.relative_to(root))

        sample_unit_id = f"wilddeepfake::{folder}::{archive_id}::{inner_unit}"

        rows.append({
            "frame_path": str(p),
            "image_path": str(p),
            "label": int(label),
            "label_name": label_name,
            "split": split,
            "sample_unit_id": sample_unit_id,
            "group_id": sample_unit_id,
            "video_id": sample_unit_id,
            "clip_id": sample_unit_id,
            "method": "wilddeepfake",
            "domain": "wilddeepfake",
            "dataset_partition": f"wilddeepfake_{folder}",
            "top_level_folder": folder,
        })

df = pd.DataFrame(rows)

print("\nRaw rows before validation split:", len(df))
print("\nCounts before validation split:")
display(pd.crosstab(df["split"], df["label"]))

print("\nUnique groups before validation split:")
display(df.groupby(["split", "label"])["sample_unit_id"].nunique().unstack(fill_value=0))

# ============================================================
# Create validation split from training groups only
# ============================================================

print("\n" + "=" * 100)
print("Creating val split from train sample units only")
print("=" * 100)

rng = np.random.default_rng(SEED)

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

val_units = []

for label, g in train_df.groupby("label"):
    units = np.array(sorted(g["sample_unit_id"].unique()))
    rng.shuffle(units)

    n_val = max(1, int(round(VAL_FRACTION * len(units))))
    selected = units[:n_val].tolist()
    val_units.extend(selected)

    print(f"label={label} train_units={len(units)} val_units={len(selected)}")

val_units = set(val_units)
train_df.loc[train_df["sample_unit_id"].isin(val_units), "split"] = "val"

df_final = pd.concat([train_df, test_df], ignore_index=True)

# ============================================================
# Safety checks
# ============================================================

print("\n" + "=" * 100)
print("Safety checks")
print("=" * 100)

print("Final counts by split and label:")
display(pd.crosstab(df_final["split"], df_final["label"]))

print("\nUnique sample units by split and label:")
display(df_final.groupby(["split", "label"])["sample_unit_id"].nunique().unstack(fill_value=0))

required_splits = ["train", "val", "test"]
for split_name in required_splits:
    labels = set(df_final[df_final["split"] == split_name]["label"].unique().tolist())
    if labels != {0, 1}:
        raise RuntimeError(f"Split {split_name} does not contain both classes. Found labels: {labels}")

duplicate_path_count = int(df_final["image_path"].duplicated().sum())
print("\nDuplicate image path count:", duplicate_path_count)
if duplicate_path_count > 0:
    dup_path = OUT_DIR / "ERROR_wilddeepfake_duplicate_paths.csv"
    df_final[df_final["image_path"].duplicated(keep=False)].to_csv(dup_path, index=False)
    raise RuntimeError(f"Duplicate paths found. Saved to: {dup_path}")

path_leak_count = int((df_final.groupby("image_path")["split"].nunique() > 1).sum())
print("Image path split leakage count:", path_leak_count)
if path_leak_count > 0:
    raise RuntimeError("Image path leakage found.")

group_leak_count = int((df_final.groupby("sample_unit_id")["split"].nunique() > 1).sum())
print("Group/sample unit split leakage count:", group_leak_count)
if group_leak_count > 0:
    raise RuntimeError("Group leakage found.")

# Check a sample of paths instead of every image to keep this fast.
sample_paths = df_final["image_path"].sample(min(5000, len(df_final)), random_state=SEED).tolist()
missing_sample = [p for p in sample_paths if not Path(p).exists()]
print("Missing image files in 5000-row sample:", len(missing_sample))
if missing_sample:
    raise FileNotFoundError(f"First missing image: {missing_sample[0]}")

# ============================================================
# Save manifests
# ============================================================

canonical_path = OUT_DIR / "wilddeepfake_manifest_canonical.csv"
train_path = OUT_DIR / "wilddeepfake_train_manifest.csv"
val_path = OUT_DIR / "wilddeepfake_val_manifest.csv"
test_path = OUT_DIR / "wilddeepfake_test_manifest_canonical.csv"
audit_path = OUT_DIR / "wilddeepfake_manifest_audit.csv"

df_final.to_csv(canonical_path, index=False)
df_final[df_final["split"] == "train"].to_csv(train_path, index=False)
df_final[df_final["split"] == "val"].to_csv(val_path, index=False)
df_final[df_final["split"] == "test"].to_csv(test_path, index=False)

audit_df = pd.DataFrame([{
    "canonical_manifest": str(canonical_path),
    "total_rows": len(df_final),
    "train_rows": int((df_final["split"] == "train").sum()),
    "val_rows": int((df_final["split"] == "val").sum()),
    "test_rows": int((df_final["split"] == "test").sum()),
    "real_rows": int((df_final["label"] == 0).sum()),
    "fake_rows": int((df_final["label"] == 1).sum()),
    "duplicate_path_count": duplicate_path_count,
    "path_leakage_count": path_leak_count,
    "group_leakage_count": group_leak_count,
    "val_fraction": VAL_FRACTION,
    "seed": SEED,
}])
audit_df.to_csv(audit_path, index=False)

print("\n" + "=" * 100)
print("Saved WildDeepfake canonical manifests")
print("=" * 100)

for p in [canonical_path, train_path, val_path, test_path, audit_path]:
    print("FOUND" if p.exists() else "MISSING", p)

print("\nReady for training:")
print(canonical_path)

Checking extracted folders
real_train   exists=True extracted_archives=371 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_train
fake_train   exists=True extracted_archives=592 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_train
real_test    exists=True extracted_archives=42 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/real_test
fake_test    exists=True extracted_archives=115 path=/home/jovyan/Deepfake_Low Resource_Low Resolution/WildDeepfake/deepfake_in_the_wild/extracted/fake_test

Building manifest rows
real_train   images=381877


Manifest real_train: 100%|██████████| 381877/381877 [00:02<00:00, 162825.75it/s]


fake_train   images=632561


Manifest fake_train: 100%|██████████| 632561/632561 [00:03<00:00, 161090.34it/s]


real_test    images=58659


Manifest real_test: 100%|██████████| 58659/58659 [00:00<00:00, 172063.92it/s]


... (7 outputs left out)

label,0,1
split,,
test,58659,107003
train,346013,563531
val,35864,69030



Unique sample units by split and label:


label,0,1
split,,
test,396,410
train,3069,2789
val,341,310



Duplicate image path count: 0
Image path split leakage count: 0
Group/sample unit split leakage count: 0
Missing image files in 5000-row sample: 0

Saved WildDeepfake canonical manifests
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_train_manifest.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_val_manifest.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_test_manifest_canonical.csv
FOUND /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_audit.csv

Ready for training:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv


In [34]:
# ============================================================
# Fix WildDeepfake manifests for FreqDebias
# Removes image_path because freqdebias_ffpp.py creates it internally.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
WILD_OUT = PROJECT_ROOT / "outputs_wilddeepfake"

FULL_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical.csv"
SAFE_FULL_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical_freqdebias_safe.csv"

SMOKE_MANIFEST = WILD_OUT / "wilddeepfake_manifest_smoke.csv"
SAFE_SMOKE_MANIFEST = WILD_OUT / "wilddeepfake_manifest_smoke_freqdebias_safe.csv"

if not FULL_MANIFEST.exists():
    raise FileNotFoundError(FULL_MANIFEST)

df = pd.read_csv(FULL_MANIFEST)

# Make sure frame_path exists.
if "frame_path" not in df.columns:
    if "image_path" in df.columns:
        df["frame_path"] = df["image_path"].astype(str)
    else:
        raise ValueError("Manifest needs frame_path or image_path.")

# Remove image_path because FreqDebias script renames frame_path -> image_path internally.
if "image_path" in df.columns:
    df = df.drop(columns=["image_path"])

# Required compatibility columns.
required_cols = ["frame_path", "split", "label", "label_name", "video_id"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column after cleanup: {c}")

df.to_csv(SAFE_FULL_MANIFEST, index=False)

print("Saved full safe manifest:")
print(SAFE_FULL_MANIFEST)

print("\nColumns:")
print(list(df.columns))

print("\nCounts:")
display(pd.crosstab(df["split"], df["label"]))

# Rebuild smoke manifest from the safe full manifest.
SMOKE_UNITS_PER_SPLIT_LABEL = {
    ("train", 0): 80,
    ("train", 1): 80,
    ("val", 0): 20,
    ("val", 1): 20,
    ("test", 0): 20,
    ("test", 1): 20,
}

parts = []
rng = np.random.default_rng(42)

for (split_name, label), n_units in SMOKE_UNITS_PER_SPLIT_LABEL.items():
    g = df[(df["split"] == split_name) & (df["label"] == label)].copy()
    units = np.array(sorted(g["video_id"].astype(str).unique()))
    rng.shuffle(units)
    selected = set(units[:min(n_units, len(units))])
    parts.append(g[g["video_id"].astype(str).isin(selected)])

smoke_df = pd.concat(parts, ignore_index=True)
smoke_df = smoke_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

# Safety: no image_path column.
if "image_path" in smoke_df.columns:
    smoke_df = smoke_df.drop(columns=["image_path"])

smoke_df.to_csv(SAFE_SMOKE_MANIFEST, index=False)

print("\nSaved smoke safe manifest:")
print(SAFE_SMOKE_MANIFEST)

print("\nSmoke counts:")
display(pd.crosstab(smoke_df["split"], smoke_df["label"]))

print("\nSmoke unique videos:")
display(smoke_df.groupby(["split", "label"])["video_id"].nunique().unstack(fill_value=0))

print("\nReady for FreqDebias smoke training.")

Saved full safe manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe.csv

Columns:
['frame_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'clip_id', 'method', 'domain', 'dataset_partition', 'top_level_folder']

Counts:


label,0,1
split,,
test,58659,107003
train,346013,563531
val,35864,69030



Saved smoke safe manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_smoke_freqdebias_safe.csv

Smoke counts:


label,0,1
split,,
test,4160,4148
train,8859,13312
val,2364,3449



Smoke unique videos:


label,0,1
split,,
test,20,20
train,80,80
val,20,20



Ready for FreqDebias smoke training.


In [11]:
# ============================================================
# STEP 5: Launch full FreqDebias training on WildDeepfake
# Uses the safe manifest with frame_path only.
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe.csv"
OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train"
LOG_PATH = OUT_DIR / "train_full.log"
PID_PATH = OUT_DIR / "train_full.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not WILD_MANIFEST.exists():
    raise FileNotFoundError(f"Missing safe WildDeepfake manifest: {WILD_MANIFEST}")

cmd = [
    sys.executable,
    "freqdebias_ffpp.py",
    "--metadata_csv", str(WILD_MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--epochs", "10",
    "--batch_size", "16",
    "--image_size", "256",
    "--num_workers", "0",
    "--amp",
]

print("Launching full FreqDebias WildDeepfake training")
print("Command:")
print(" ".join(cmd))
print("\nOutput dir:", OUT_DIR)
print("Log path:", LOG_PATH)

log_file = open(LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

PID_PATH.write_text(str(proc.pid))

print("\nStarted process PID:", proc.pid)
print("Saved PID:", PID_PATH)
print("\nTraining is now running in the background.")
print("Run the next monitoring cell to check progress.")

Launching full FreqDebias WildDeepfake training
Command:
/opt/conda/bin/python freqdebias_ffpp.py --metadata_csv /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_freqdebias_safe.csv --out_dir /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train --epochs 10 --batch_size 16 --image_size 256 --num_workers 0 --amp

Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train
Log path: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/train_full.log

Started process PID: 715
Saved PID: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/train_full.pid

Training is now running in the background.
Run the next monitoring cell to check progress.


In [10]:
# ============================================================
# Monitor FreqDebias WildDeepfake training
# Re-run this cell whenever you want to check status.
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd
import os

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train"
LOG_PATH = OUT_DIR / "train_full.log"
PID_PATH = OUT_DIR / "train_full.pid"

print("=" * 100)
print("FreqDebias WildDeepfake training status")
print("=" * 100)

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)

    ps = subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,etime,stat,%cpu,%mem,cmd"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(ps.stdout)
else:
    print("No PID file found:", PID_PATH)

print("\n" + "=" * 100)
print("GPU status")
print("=" * 100)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(gpu.stdout if gpu.stdout.strip() else "No GPU compute process found.")

print("\n" + "=" * 100)
print("Output files")
print("=" * 100)

for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "freqdebias_best.pt",
    "freqdebias_latest.pt",
    "training_period_results.csv",
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_inference_time.csv",
]:
    p = OUT_DIR / name
    print("FOUND  " if p.exists() else "MISSING", p)

print("\n" + "=" * 100)
print("Training history")
print("=" * 100)

for name in ["training_period_results.csv", "training_history.csv"]:
    p = OUT_DIR / name
    if p.exists():
        df = pd.read_csv(p)
        print("History file:", p)
        display(df.tail())
        break
else:
    print("No training history CSV yet.")

print("\n" + "=" * 100)
print("Last 80 log lines")
print("=" * 100)

if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-80:]:
        print(line)
else:
    print("No log found yet:", LOG_PATH)

FreqDebias WildDeepfake training status
PID: 500
    PID     ELAPSED STAT %CPU %MEM CMD
    500       13:11 Z     1.9  0.0 [python] <defunct>


GPU status
413, python, 992


Output files
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/best_checkpoint.pt
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/latest_checkpoint.pt
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/freqdebias_best.pt
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/freqdebias_latest.pt
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/training_period_results.csv
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/test_metrics.csv
MISSING /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train/test_frame_predi

,phase,epoch,train_loss,train_acc,val_Accuracy,val_AUPRC,val_AUCROC,val_EER (Equal Error Rate),val_Precision (0),val_Recall (0),val_F1 (0),val_Precision (1),val_Recall (1),val_F1 (1),val_Uncertain Rate (%)
0,pretrain,1,0.136254,0.948886,0.921323,0.976898,0.976851,0.06941,0.948014,0.899102,0.92291,0.894973,0.945766,0.919669,2.736175



Last 80 log lines
Ignoring notebook/system args: ['--epochs', '10', '--amp']

FreqDebias FF++ regular experiment
Device: cuda
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_freqdebias_wilddeepfake_train
Traceback (most recent call last):
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 1007, in <module>
    main()
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 873, in main
    df = load_ffpp_metadata(args.metadata_csv)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/freqdebias_ffpp.py", line 140, in load_ffpp_metadata
    df = df[df["frame_path"].apply(lambda p: isinstance(p, str) and os.path.exists(p))]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/series.py", line 5084, in apply
    ).apply()
      ^^^^^^^
  File "/opt/conda/lib/pytho

In [ ]:
# ============================================================
# Monitor current FreqDebias run more clearly
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd
import time
import os

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_freqdebias_wilddeepfake_train"
LOG_PATH = OUT_DIR / "train_full.log"
PID_PATH = OUT_DIR / "train_full.pid"

print("=" * 100)
print("Process check")
print("=" * 100)

pid = PID_PATH.read_text().strip() if PID_PATH.exists() else None
print("Saved PID:", pid)

if pid:
    ps = subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(ps.stdout)

print("\n" + "=" * 100)
print("All Python / GPU-related processes")
print("=" * 100)

proc_check = subprocess.run(
    ["bash", "-lc", "ps aux | egrep 'freqdebias|python' | grep -v egrep | head -30"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(proc_check.stdout)

print("\n" + "=" * 100)
print("GPU status")
print("=" * 100)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(gpu.stdout)

print("\n" + "=" * 100)
print("Output files")
print("=" * 100)

for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "freqdebias_best.pt",
    "freqdebias_latest.pt",
    "training_history.csv",
    "training_period_results.csv",
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_inference_time.csv",
    "train_full.log",
]:
    p = OUT_DIR / name
    if p.exists():
        age_min = (time.time() - p.stat().st_mtime) / 60
        size_mb = p.stat().st_size / 1024**2
        print(f"FOUND   {name:35s} size_mb={size_mb:8.3f} age_min={age_min:8.1f}")
    else:
        print(f"MISSING {name}")

print("\n" + "=" * 100)
print("Training history")
print("=" * 100)

for name in ["training_history.csv", "training_period_results.csv"]:
    p = OUT_DIR / name
    if p.exists():
        hist = pd.read_csv(p)
        print("History file:", p)
        print("Rows:", len(hist))
        display(hist.tail(10))
        break
else:
    print("No history CSV yet.")

print("\n" + "=" * 100)
print("Log tail")
print("=" * 100)

if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-100:]:
        print(line)
else:
    print("No log file found.")

In [14]:
!pip install pillow torchvision scikit-learn

  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.25.0-py3-none-any.whl (467 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
WILD_OUT = PROJECT_ROOT / "outputs_wilddeepfake"

FULL_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical.csv"
GEND_SAFE_MANIFEST = WILD_OUT / "wilddeepfake_manifest_canonical_gend_safe.csv"

df = pd.read_csv(FULL_MANIFEST)

print("Original columns:")
print(list(df.columns))
print("Rows:", len(df))

if "frame_path" not in df.columns:
    if "image_path" in df.columns:
        df["frame_path"] = df["image_path"].astype(str)
    else:
        raise ValueError("Need frame_path or image_path column.")

# Critical fix: remove image_path.
# GenD creates image_path internally from frame_path.
if "image_path" in df.columns:
    df = df.drop(columns=["image_path"])

required = ["frame_path", "split", "label", "label_name", "video_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df.to_csv(GEND_SAFE_MANIFEST, index=False)

print("\nSaved GenD-safe manifest:")
print(GEND_SAFE_MANIFEST)

print("\nSafe columns:")
print(list(df.columns))

print("\nCounts:")
display(pd.crosstab(df["split"], df["label"]))

from pathlib import Path
import subprocess
import sys
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

SCRIPT = PROJECT_ROOT / "gend_reimpl_ffpp.py"
MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv"
OUT_DIR = PROJECT_ROOT / "outputs_gend_wilddeepfake_train"
LOG_PATH = OUT_DIR / "train_gend_nohup.log"
PID_PATH = OUT_DIR / "train_gend_nohup.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(SCRIPT),
    "--metadata_csv", str(MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--epochs", "10",
    "--batch_size", "16",
    "--frames_per_video", "32",
    "--image_size", "224",
    "--num_workers", "0",
    "--amp",
]

print("Launching GenD with safe manifest")
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Log:", LOG_PATH)

existing = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'gend_reimpl_ffpp.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing:
    print("\nA GenD process already exists. Not launching duplicate:")
    print(existing)
else:
    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("\nLaunched GenD PID:", pid)
    print("PID saved to:", PID_PATH)

Original columns:
['frame_path', 'image_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'clip_id', 'method', 'domain', 'dataset', 'dataset_partition', 'top_level_folder']
Rows: 1180100

Saved GenD-safe manifest:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv

Safe columns:
['frame_path', 'label', 'label_name', 'split', 'sample_unit_id', 'group_id', 'video_id', 'clip_id', 'method', 'domain', 'dataset', 'dataset_partition', 'top_level_folder']

Counts:


label,0,1
split,,
test,58659,107003
train,346013,563531
val,35864,69030


Launching GenD with safe manifest
Command:
/opt/conda/bin/python '/home/jovyan/Deepfake_Low Resource_Low Resolution/gend_reimpl_ffpp.py' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_train' --epochs 10 --batch_size 16 --frames_per_video 32 --image_size 224 --num_workers 0 --amp
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_gend_wilddeepfake_train/train_gend_nohup.log


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_gend_wilddeepfake_train"
LOG_PATH = OUT_DIR / "train_gend_nohup.log"
PID_PATH = OUT_DIR / "train_gend_nohup.pid"

print("=" * 100)
print("GenD WildDeepfake status")
print("=" * 100)

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    ps = subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,etime,stat,%cpu,%mem,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(ps.stdout)
else:
    print("No PID file found.")

print("\nGPU:")
gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(gpu.stdout if gpu.stdout.strip() else "No GPU process found.")

print("\nOutput files:")
for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "gend_best.pt",
    "gend_latest.pt",
    "training_history.csv",
    "training_period_results.csv",
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_inference_time.csv",
]:
    p = OUT_DIR / name
    print("FOUND  " if p.exists() else "MISSING", p)

print("\nHistory:")
for name in ["training_history.csv", "training_period_results.csv"]:
    p = OUT_DIR / name
    if p.exists():
        display(pd.read_csv(p).tail(10))
        break

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-100:]:
        print(line)
else:
    print("No log yet.")

In [15]:
from pathlib import Path

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"

print("DFD-HR root:", DFDHR_ROOT)
print("Exists:", DFDHR_ROOT.exists())

patterns = ["*train*.py", "*main*.py", "*run*.py", "*.py"]

seen = set()
for pattern in patterns:
    for p in sorted(DFDHR_ROOT.rglob(pattern)):
        if p not in seen:
            seen.add(p)
            print(p.relative_to(DFDHR_ROOT))

DFD-HR root: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR
Exists: True
training/train.py
training/trainer/trainer.py
run_dfd_hr_df40.py
training/.ipynb_checkpoints/test-checkpoint.py
training/dataset/__init__.py
training/dataset/abstract_dataset.py
training/dataset/albu.py
training/detectors/__init__.py
training/detectors/base_detector.py
training/detectors/dfd_hr_detector.py
training/detectors/utils/__init__.py
training/detectors/utils/core_query_loss.py
training/detectors/utils/mod.py
training/detectors/utils/moe_adapter.py
training/detectors/utils/utils.py
training/logger.py
training/loss/__init__.py
training/loss/abstract_loss_func.py
training/loss/cross_entropy_loss.py
training/metrics/__init__.py
training/metrics/base_metrics_class.py
training/metrics/registry.py
training/metrics/utils.py
training/optimizor/LinearLR.py
training/optimizor/SAM.py
training/optimizor/__init__.py
training/test.py
training/trainer/__init__.py


In [16]:
# ============================================================
# Inspect DFD-HR training interface
# ============================================================

from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"

scripts = [
    DFDHR_ROOT / "training/train.py",
    DFDHR_ROOT / "training/test.py",
    DFDHR_ROOT / "run_dfd_hr_df40.py",
]

for script in scripts:
    print("\n" + "=" * 120)
    print("SCRIPT:", script)
    print("=" * 120)

    if not script.exists():
        print("MISSING")
        continue

    # Try --help first.
    try:
        out = subprocess.run(
            [sys.executable, str(script), "--help"],
            cwd=str(DFDHR_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=30,
        )
        print(out.stdout[:8000])
    except Exception as e:
        print("Help failed:", repr(e))

    # Show argparse/config-related lines.
    print("\n--- argparse/config/import clues ---")
    try:
        text = script.read_text(errors="replace").splitlines()
        for i, line in enumerate(text, start=1):
            low = line.lower()
            if (
                "argparse" in low
                or "add_argument" in low
                or "parser." in low
                or "yaml" in low
                or "config" in low
                or "dataset" in low
                or "detector" in low
                or "train" in low
                or "checkpoint" in low
            ):
                print(f"{i:04d}: {line}")
    except Exception as e:
        print("Could not read script:", repr(e))


SCRIPT: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/train.py
Traceback (most recent call last):
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/train.py", line 9, in <module>
    import cv2
ModuleNotFoundError: No module named 'cv2'


--- argparse/config/import clues ---
0004: # description: training code.
0007: import argparse
0013: import yaml
0032: from trainer.trainer import Trainer
0033: from detectors import DETECTOR
0037: from dataset.abstract_dataset import DeepfakeAbstractBaseDataset
0039: parser = argparse.ArgumentParser(description='Process some paths.')
0040: parser.add_argument('--detector_path', type=str,
0041:                     default='./training/config/detector/dfd_hr.yaml',
0042:                     help='path to detector YAML file')
0043: parser.add_argument("--train_dataset", nargs="+")
0044: parser.add_argument("--test_dataset", nargs="+")
0045: parser.add_argument('--no-save_ckpt', dest='save_ckpt', action='store_

In [1]:
# ============================================================
# DFD-HR Manifest Training: WildDeepfake -> DF40 setup
# One-cell script creation + launch
#
# Default: smoke run
# After smoke succeeds, change:
#   RUN_MODE = "smoke"
# to:
#   RUN_MODE = "full"
# ============================================================

from pathlib import Path
import subprocess
import sys
import shlex
import textwrap

# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

RUN_MODE = "full"   # "smoke" or "full"

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"
WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

TRAIN_SCRIPT = PROJECT_ROOT / "train_dfdhr_manifest_wilddeepfake.py"

if RUN_MODE == "smoke":
    OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_smoke"
    EPOCHS = 1
    BATCH_SIZE = 2
    IMAGE_SIZE = 448
    FRAMES_PER_VIDEO = 8
    LIMIT_TRAIN_VIDEOS = 40
    LIMIT_VAL_VIDEOS = 20
else:
    OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train"
    EPOCHS = 5
    BATCH_SIZE = 4
    IMAGE_SIZE = 448
    FRAMES_PER_VIDEO = 32
    LIMIT_TRAIN_VIDEOS = None
    LIMIT_VAL_VIDEOS = None

LOG_PATH = OUT_DIR / f"train_dfdhr_{RUN_MODE}.log"
PID_PATH = OUT_DIR / f"train_dfdhr_{RUN_MODE}.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Install missing dependency
# ------------------------------------------------------------

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "opencv-python-headless"],
    check=True,
)

# ------------------------------------------------------------
# Write manifest-based DFD-HR trainer
# ------------------------------------------------------------

script_code = r'''
import argparse
import os
import sys
import time
import random
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
)


def parse_args():
    p = argparse.ArgumentParser("Train DFD-HR from manifest")
    p.add_argument("--repo_root", type=str, required=True)
    p.add_argument("--metadata_csv", type=str, required=True)
    p.add_argument("--out_dir", type=str, required=True)

    p.add_argument("--detector_yaml", type=str, default="training/config/detector/dfd_hr.yaml")
    p.add_argument("--train_config", type=str, default="training/config/train_config.yaml")

    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--batch_size", type=int, default=4)
    p.add_argument("--image_size", type=int, default=448)
    p.add_argument("--frames_per_video", type=int, default=32)
    p.add_argument("--num_workers", type=int, default=0)

    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--weight_decay", type=float, default=1e-4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--amp", action="store_true")

    p.add_argument("--limit_train_videos", type=int, default=None)
    p.add_argument("--limit_val_videos", type=int, default=None)

    return p.parse_args()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_yaml(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r") as f:
        x = yaml.safe_load(f)
    return x or {}


def normalize_manifest(path):
    df = pd.read_csv(path)

    if "frame_path" not in df.columns:
        if "image_path" in df.columns:
            df["frame_path"] = df["image_path"].astype(str)
        else:
            raise ValueError("Manifest must contain frame_path or image_path.")

    if "video_id" not in df.columns:
        if "sample_unit_id" in df.columns:
            df["video_id"] = df["sample_unit_id"].astype(str)
        elif "group_id" in df.columns:
            df["video_id"] = df["group_id"].astype(str)
        else:
            df["video_id"] = df["frame_path"].apply(lambda x: str(Path(x).parent))

    required = ["frame_path", "split", "label", "video_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing manifest columns: {missing}")

    df["frame_path"] = df["frame_path"].astype(str)
    df["video_id"] = df["video_id"].astype(str)
    df["split"] = df["split"].astype(str)
    df["label"] = df["label"].astype(int)

    return df


def sample_frames_per_video(df, frames_per_video, limit_videos=None, seed=42):
    rng = np.random.default_rng(seed)
    vids = np.array(sorted(df["video_id"].unique()))

    if limit_videos is not None:
        rng.shuffle(vids)
        vids = vids[:min(limit_videos, len(vids))]

    parts = []

    for vid in vids:
        g = df[df["video_id"] == vid].sort_values("frame_path").reset_index(drop=True)

        if len(g) <= frames_per_video:
            parts.append(g)
        else:
            idx = np.linspace(0, len(g) - 1, frames_per_video).round().astype(int)
            parts.append(g.iloc[idx].copy())

    if not parts:
        return df.iloc[0:0].copy()

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


class ManifestImageDataset(Dataset):
    def __init__(self, df, image_size=448, train=False):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size

        if train:
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["frame_path"]
        label = int(row["label"])

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.image_size, self.image_size))

        img = self.tf(img)

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
        }


def extract_logits(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, dict):
        for key in [
            "logits", "cls", "pred", "prediction", "output", "out",
            "prob", "y_pred", "label", "final_output"
        ]:
            if key in output and torch.is_tensor(output[key]):
                return output[key]

        for value in output.values():
            if torch.is_tensor(value):
                return value

    if isinstance(output, (list, tuple)):
        for value in output:
            if torch.is_tensor(value):
                return value

        for value in output:
            try:
                return extract_logits(value)
            except Exception:
                pass

    raise RuntimeError(f"Could not extract logits from output type: {type(output)}")


def forward_model(model, images, labels):
    data_dict = {
        "image": images,
        "label": labels,
    }

    attempts = [
        lambda: model(data_dict),
        lambda: model(data_dict, inference=False),
        lambda: model(images),
    ]

    last_error = None

    for fn in attempts:
        try:
            output = fn()
            logits = extract_logits(output)

            if logits.ndim == 1:
                logits = logits.view(-1, 1)

            if logits.ndim > 2:
                logits = logits.view(logits.shape[0], -1)

            return logits

        except Exception as e:
            last_error = e

    raise RuntimeError(f"DFD-HR forward failed. Last error: {repr(last_error)}")


def compute_loss(logits, labels):
    if logits.shape[1] == 1:
        return F.binary_cross_entropy_with_logits(logits[:, 0], labels.float())

    return F.cross_entropy(logits[:, :2], labels)


def fake_scores_from_logits(logits):
    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])

    return torch.softmax(logits[:, :2], dim=1)[:, 1]


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    y_true = []
    y_score = []
    losses = []

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = forward_model(model, images, labels)
        loss = compute_loss(logits, labels)
        score = fake_scores_from_logits(logits)

        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_score.extend(score.detach().cpu().numpy().tolist())
        losses.append(float(loss.detach().cpu()))

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    out = {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "AUPRC": average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "AUCROC": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "Precision (0)": precision[0],
        "Recall (0)": recall[0],
        "F1 (0)": f1[0],
        "Precision (1)": precision[1],
        "Recall (1)": recall[1],
        "F1 (1)": f1[1],
    }

    return out


def main():
    args = parse_args()
    set_seed(args.seed)

    repo = Path(args.repo_root).resolve()
    out_dir = Path(args.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    sys.path.insert(0, str(repo / "training"))

    from detectors import DETECTOR

    detector_cfg = load_yaml(repo / args.detector_yaml)
    train_cfg = load_yaml(repo / args.train_config)

    config = {}
    config.update(train_cfg)
    config.update(detector_cfg)

    config["cuda"] = torch.cuda.is_available()
    config["manualSeed"] = args.seed
    config["train_batchSize"] = args.batch_size
    config["test_batchSize"] = args.batch_size
    config["workers"] = args.num_workers

    detector_name = config.get("name", config.get("model_name", "dfd_hr"))
    config["model_name"] = detector_name

    print("=" * 100)
    print("DFD-HR manifest training")
    print("=" * 100)
    print("Repo:", repo)
    print("Detector:", detector_name)
    print("Manifest:", args.metadata_csv)
    print("Output:", out_dir)
    print("Epochs:", args.epochs)
    print("Batch size:", args.batch_size)
    print("Image size:", args.image_size)
    print("Frames per video:", args.frames_per_video)

    df = normalize_manifest(args.metadata_csv)

    train_df = df[df["split"] == "train"].copy()
    val_df = df[df["split"] == "val"].copy()

    train_df = sample_frames_per_video(
        train_df,
        frames_per_video=args.frames_per_video,
        limit_videos=args.limit_train_videos,
        seed=args.seed,
    )

    val_df = sample_frames_per_video(
        val_df,
        frames_per_video=args.frames_per_video,
        limit_videos=args.limit_val_videos,
        seed=args.seed,
    )

    print("\nTrain rows:", len(train_df), "videos:", train_df["video_id"].nunique())
    print("Val rows:", len(val_df), "videos:", val_df["video_id"].nunique())

    print("\nTrain counts:")
    print(pd.crosstab(train_df["split"], train_df["label"]))

    print("\nVal counts:")
    print(pd.crosstab(val_df["split"], val_df["label"]))

    train_df.to_csv(out_dir / "train_manifest_used.csv", index=False)
    val_df.to_csv(out_dir / "val_manifest_used.csv", index=False)

    train_ds = ManifestImageDataset(train_df, image_size=args.image_size, train=True)
    val_ds = ManifestImageDataset(val_df, image_size=args.image_size, train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    try:
        model_cls = DETECTOR[detector_name]
    except Exception:
        model_cls = DETECTOR["dfd_hr"]

    model = model_cls(config).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")

    history = []
    best_score = -1.0

    for epoch in range(1, args.epochs + 1):
        model.train()

        train_losses = []
        correct = 0
        total = 0
        start = time.time()

        for step, batch in enumerate(train_loader, start=1):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=args.amp and device.type == "cuda"):
                logits = forward_model(model, images, labels)
                loss = compute_loss(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(float(loss.detach().cpu()))

            with torch.no_grad():
                scores = fake_scores_from_logits(logits)
                preds = (scores >= 0.5).long()
                correct += int((preds == labels).sum().detach().cpu())
                total += int(labels.numel())

            if step % 50 == 0 or step == len(train_loader):
                print(
                    f"epoch={epoch} step={step}/{len(train_loader)} "
                    f"loss={np.mean(train_losses):.4f} "
                    f"acc={correct / max(total, 1):.4f}",
                    flush=True,
                )

        val_metrics = evaluate(model, val_loader, device)

        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        train_acc = correct / max(total, 1)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "elapsed_min": (time.time() - start) / 60,
        }

        history.append(row)
        pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "config": config,
            "args": vars(args),
            "history": history,
        }

        torch.save(ckpt, out_dir / "latest_checkpoint.pt")

        score = val_metrics.get("AUCROC", np.nan)
        if np.isnan(score):
            score = val_metrics.get("Balanced Accuracy", -1.0)

        if score > best_score:
            best_score = score
            torch.save(ckpt, out_dir / "best_checkpoint.pt")

        print("\n" + "-" * 100)
        print(f"Epoch {epoch}/{args.epochs}")
        print("train_loss:", train_loss)
        print("train_acc:", train_acc)
        print("val_metrics:", val_metrics)
        print("best_score:", best_score)
        print("-" * 100, flush=True)

    print("\nTraining complete.")
    print("Best checkpoint:", out_dir / "best_checkpoint.pt")
    print("Latest checkpoint:", out_dir / "latest_checkpoint.pt")


if __name__ == "__main__":
    main()
'''

TRAIN_SCRIPT.write_text(script_code)

print("Wrote trainer script:")
print(TRAIN_SCRIPT)

# ------------------------------------------------------------
# Launch training
# ------------------------------------------------------------

cmd = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--repo_root", str(DFDHR_ROOT),
    "--metadata_csv", str(WILD_MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--image_size", str(IMAGE_SIZE),
    "--frames_per_video", str(FRAMES_PER_VIDEO),
    "--num_workers", "0",
    "--amp",
]

if LIMIT_TRAIN_VIDEOS is not None:
    cmd += ["--limit_train_videos", str(LIMIT_TRAIN_VIDEOS)]

if LIMIT_VAL_VIDEOS is not None:
    cmd += ["--limit_val_videos", str(LIMIT_VAL_VIDEOS)]

print("\nLaunching DFD-HR training")
print("Run mode:", RUN_MODE)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Output dir:", OUT_DIR)
print("Log:", LOG_PATH)

existing = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'train_dfdhr_manifest_wilddeepfake.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing:
    print("\nA DFD-HR manifest training process already exists. Not launching duplicate:")
    print(existing)
else:
    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("\nLaunched DFD-HR PID:", pid)
    print("PID saved to:", PID_PATH)

print("\nTo monitor:")
print(f"tail -f {shlex.quote(str(LOG_PATH))}")

Wrote trainer script:
/home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py

Launching DFD-HR training
Run mode: full
Command:
/opt/conda/bin/python '/home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py' --repo_root '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train' --epochs 5 --batch_size 4 --image_size 448 --frames_per_video 32 --num_workers 0 --amp
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train/train_dfdhr_full.log

Launched DFD-HR PID: 1524
PID saved to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train/train_dfdhr_full.pid

To

In [9]:
# ============================================================
# DFD-HR WildDeepfake smoke run WITHOUT AMP
# Fixes dtype error:
#   Index put requires source/destination dtypes match, got Half vs Float
# ============================================================

from pathlib import Path
import subprocess
import sys
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"
SCRIPT = PROJECT_ROOT / "train_dfdhr_manifest_wilddeepfake.py"
WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_smoke_no_amp"
LOG_PATH = OUT_DIR / "train_dfdhr_smoke_no_amp.log"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stop stale DFD-HR wrapper runs only.
subprocess.run(
    ["bash", "-lc", "pkill -f 'train_dfdhr_manifest_wilddeepfake.py' || true"],
    text=True,
)

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--repo_root", str(DFDHR_ROOT),
    "--metadata_csv", str(WILD_MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--epochs", "1",
    "--batch_size", "2",
    "--image_size", "448",
    "--frames_per_video", "8",
    "--num_workers", "0",
    "--limit_train_videos", "8",
    "--limit_val_videos", "4",
    # IMPORTANT: no --amp
]

print("Running DFD-HR smoke WITHOUT AMP:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Log:", LOG_PATH)

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line.rstrip())
        log.write(line)

    proc.wait()

print("\nReturn code:", proc.returncode)

if proc.returncode != 0:
    print("\nDFD-HR no-AMP smoke failed. Last 120 log lines:")
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-120:]:
        print(line)
    raise RuntimeError("DFD-HR no-AMP smoke failed.")

print("\nDFD-HR no-AMP smoke completed.")

for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "train_manifest_used.csv",
    "val_manifest_used.csv",
]:
    p = OUT_DIR / name
    print("FOUND  " if p.exists() else "MISSING", p)

history_path = OUT_DIR / "training_history.csv"
if history_path.exists():
    import pandas as pd
    display(pd.read_csv(history_path))

Running DFD-HR smoke WITHOUT AMP:
/opt/conda/bin/python -u '/home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py' --repo_root '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_smoke_no_amp' --epochs 1 --batch_size 2 --image_size 448 --frames_per_video 8 --num_workers 0 --limit_train_videos 8 --limit_val_videos 4
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_smoke_no_amp/train_dfdhr_smoke_no_amp.log
DFD-HR manifest training
Repo: /home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR
Detector: dfd_hr
Manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_

,epoch,train_loss,train_acc,val_loss,val_Accuracy,val_Balanced Accuracy,val_AUPRC,val_AUCROC,val_Precision (0),val_Recall (0),val_F1 (0),val_Precision (1),val_Recall (1),val_F1 (1),elapsed_min
0,1,0.305986,0.875,1.101536,0.5,0.5,0.457973,0.464844,0.0,0.0,0.0,0.5,1.0,0.666667,0.999727


In [ ]:
# ============================================================
# FULL DFD-HR training on WildDeepfake WITHOUT AMP
# Uses compiler env for Triton/PyTorch JIT
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"
SCRIPT = PROJECT_ROOT / "train_dfdhr_manifest_wilddeepfake.py"
WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"
LOG_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_no_amp_cc.log"
PID_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_no_amp_cc.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT.exists():
    raise FileNotFoundError(f"Missing DFD-HR trainer script: {SCRIPT}")

if not WILD_MANIFEST.exists():
    raise FileNotFoundError(f"Missing WildDeepfake manifest: {WILD_MANIFEST}")

# ------------------------------------------------------------
# Find compiler for Triton/PyTorch JIT
# ------------------------------------------------------------

gcc_path = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-gcc || which gcc || which cc"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
).stdout.strip().splitlines()[0]

gxx_result = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-g++ || which g++ || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip().splitlines()

gxx_path = gxx_result[0] if gxx_result else gcc_path

print("Using CC :", gcc_path)
print("Using CXX:", gxx_path)

# ------------------------------------------------------------
# Optional safety: do not launch duplicate DFD-HR full run
# ------------------------------------------------------------

existing = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'train_dfdhr_manifest_wilddeepfake.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing:
    print("A DFD-HR manifest training process is already running. Not launching duplicate:")
    print(existing)
else:
    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT),
        "--repo_root", str(DFDHR_ROOT),
        "--metadata_csv", str(WILD_MANIFEST),
        "--out_dir", str(OUT_DIR),
        "--epochs", "5",
        "--batch_size", "4",
        "--image_size", "448",
        "--frames_per_video", "32",
        "--num_workers", "0",
        # IMPORTANT: no --amp for DFD-HR
    ]

    env = os.environ.copy()
    env["CC"] = gcc_path
    env["CXX"] = gxx_path
    env["CUDA_HOME"] = env.get("CUDA_HOME", "/usr/local/cuda")

    print("\nLaunching FULL DFD-HR WildDeepfake training:")
    print(" ".join(shlex.quote(x) for x in cmd))
    print("Output dir:", OUT_DIR)
    print("Log:", LOG_PATH)

    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "CC=" + shlex.quote(gcc_path) + " "
        "CXX=" + shlex.quote(gxx_path) + " "
        "CUDA_HOME=" + shlex.quote(env["CUDA_HOME"]) + " "
        "nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("\nLaunched DFD-HR full training PID:", pid)
    print("PID saved to:", PID_PATH)
    print("Log:", LOG_PATH)

Using CC : /opt/conda/bin/x86_64-conda-linux-gnu-gcc
Using CXX: /opt/conda/bin/x86_64-conda-linux-gnu-g++

Launching FULL DFD-HR WildDeepfake training:
/opt/conda/bin/python -u '/home/jovyan/Deepfake_Low Resource_Low Resolution/train_dfdhr_manifest_wilddeepfake.py' --repo_root '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc' --epochs 5 --batch_size 4 --image_size 448 --frames_per_video 32 --num_workers 0
Output dir: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/train_dfdhr_wilddeepfake_no_amp_cc.log


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"
LOG_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_no_amp_cc.log"
PID_PATH = OUT_DIR / "train_dfdhr_wilddeepfake_no_amp_cc.pid"

print("=" * 100)
print("DFD-HR WildDeepfake full training status")
print("=" * 100)

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,etime,stat,%cpu,%mem,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)
else:
    print("No PID file found.")

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout)

print("\nOutput files:")
for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "train_manifest_used.csv",
    "val_manifest_used.csv",
]:
    p = OUT_DIR / name
    if p.exists():
        age_min = (time.time() - p.stat().st_mtime) / 60
        size_mb = p.stat().st_size / 1024**2
        print(f"FOUND   {name:30s} size_mb={size_mb:8.2f} age_min={age_min:8.1f}")
    else:
        print(f"MISSING {name}")

print("\nHistory:")
p = OUT_DIR / "training_history.csv"
if p.exists():
    display(pd.read_csv(p).tail(10))
else:
    print("No training_history.csv yet.")

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

In [8]:
# ============================================================
# Fix Triton/PyTorch "Failed to find C compiler" for DFD-HR
# Then rerun DFD-HR WildDeepfake smoke WITHOUT AMP
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
DFDHR_ROOT = PROJECT_ROOT / "DFD-HR"
SCRIPT = PROJECT_ROOT / "train_dfdhr_manifest_wilddeepfake.py"
WILD_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_smoke_no_amp_cc"
LOG_PATH = OUT_DIR / "train_dfdhr_smoke_no_amp_cc.log"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Checking for C/C++ compiler")
print("=" * 100)

check = subprocess.run(
    ["bash", "-lc", "which gcc || which cc || true; which g++ || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(check.stdout)

if not check.stdout.strip():
    print("\nNo compiler found. Installing conda compilers...")
    subprocess.run(
        ["bash", "-lc", "conda install -y -c conda-forge gcc_linux-64 gxx_linux-64"],
        check=True,
    )

print("\nCompiler after install:")
check2 = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-gcc || which gcc || which cc; which x86_64-conda-linux-gnu-g++ || which g++"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(check2.stdout)

# Pick compiler paths.
gcc_path = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-gcc || which gcc || which cc"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
).stdout.strip().splitlines()[0]

gxx_path = subprocess.run(
    ["bash", "-lc", "which x86_64-conda-linux-gnu-g++ || which g++ || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip().splitlines()

gxx_path = gxx_path[0] if gxx_path else gcc_path

print("Using CC :", gcc_path)
print("Using CXX:", gxx_path)

# Stop stale DFD-HR wrapper runs only.
subprocess.run(
    ["bash", "-lc", "pkill -f 'train_dfdhr_manifest_wilddeepfake.py' || true"],
    text=True,
)

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--repo_root", str(DFDHR_ROOT),
    "--metadata_csv", str(WILD_MANIFEST),
    "--out_dir", str(OUT_DIR),
    "--epochs", "1",
    "--batch_size", "2",
    "--image_size", "448",
    "--frames_per_video", "8",
    "--num_workers", "0",
    "--limit_train_videos", "8",
    "--limit_val_videos", "4",
    # IMPORTANT: still no --amp
]

env = os.environ.copy()
env["CC"] = gcc_path
env["CXX"] = gxx_path
env["CUDA_HOME"] = env.get("CUDA_HOME", "/usr/local/cuda")

print("\nRunning DFD-HR smoke with compiler configured:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Log:", LOG_PATH)

with open(LOG_PATH, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in proc.stdout:
        print(line.rstrip())
        log.write(line)

    proc.wait()

print("\nReturn code:", proc.returncode)

if proc.returncode != 0:
    print("\nDFD-HR smoke failed. Last 150 log lines:")
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-150:]:
        print(line)
    raise RuntimeError("DFD-HR smoke failed even after compiler fix.")

print("\nDFD-HR smoke completed.")

for name in [
    "best_checkpoint.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "train_manifest_used.csv",
    "val_manifest_used.csv",
]:
    p = OUT_DIR / name
    print("FOUND  " if p.exists() else "MISSING", p)

history_path = OUT_DIR / "training_history.csv"
if history_path.exists():
    import pandas as pd
    display(pd.read_csv(history_path))

Checking for C/C++ compiler


No compiler found. Installing conda compilers...
Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done




==> WARNING: A newer version of conda exists. <==
    current version: 24.5.0
    latest version: 26.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - gcc_linux-64
    - gxx_linux-64


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    binutils_impl_linux-64-2.46.1|bootstrap_h59bd682_2         3.5 MB  conda-forge
    binutils_linux-64-2.46.1   |bootstrap_h4852527_2          36 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    certifi-2026.7.22          |     pyhd8ed1ab_0         134 KB  conda-forge
    gcc_impl_linux-64-16.2.0   |       h176d5d0_4        82.5 MB  conda-forge
    gcc_linux-64-16.2.0        |       h0ab548f_1          29 KB  conda-forge
    gxx_impl_linux-64-16.2.0   |       h0d273dc_4        16.9 MB  conda-forge
    gxx_linux-64-16.2.0        |       h38dc33c_1          27 KB  conda-forge
    kernel-headers_linux-64-5.14.0|       he073ed8_3         1.3 MB  conda-forge
    ld_impl_linux-64-

,epoch,train_loss,train_acc,val_loss,val_Accuracy,val_Balanced Accuracy,val_AUPRC,val_AUCROC,val_Precision (0),val_Recall (0),val_F1 (0),val_Precision (1),val_Recall (1),val_F1 (1),elapsed_min
0,1,0.305969,0.875,1.096646,0.5,0.5,0.457973,0.464844,0.0,0.0,0.0,0.5,1.0,0.666667,1.022934


In [1]:
!tail -f '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_smoke/train_dfdhr_smoke.log'

    from detectors import DETECTOR
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/detectors/__init__.py", line 11, in <module>
    from .dfd_hr_detector import DFDHRDetector
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/detectors/dfd_hr_detector.py", line 42, in <module>
    from .utils import forward_query_loss as multiscale_forward_query_loss
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/detectors/utils/__init__.py", line 1, in <module>
    from .core_query_loss import *
  File "/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/training/detectors/utils/core_query_loss.py", line 12, in <module>
    from einops import rearrange
ModuleNotFoundError: No module named 'einops'
^C


In [ ]:
# ============================================================
# GenD TEST-ONLY evaluation: WildDeepfake -> WildDeepfake
#
# Computes in-domain results:
#   Frame metrics
#   Video/group metrics
#   EER
#   Uncertain Rate
#   Inference time
#
# Does NOT retrain.
# ============================================================

from pathlib import Path
import sys
import time
import inspect
import importlib.util
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    roc_curve,
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
GEND_SCRIPT = PROJECT_ROOT / "gend_reimpl_ffpp.py"

# WildDeepfake-trained GenD checkpoint.
GEND_OUT = PROJECT_ROOT / "outputs_gend_wilddeepfake_train"

CKPT_CANDIDATES = [
    GEND_OUT / "best_checkpoint.pt",
    GEND_OUT / "latest_checkpoint.pt",
    GEND_OUT / "gend_best.pt",
    GEND_OUT / "gend_latest.pt",
    GEND_OUT / "checkpoint_best.pt",
    GEND_OUT / "checkpoint_latest.pt",
]

CKPT_PATH = None
for p in CKPT_CANDIDATES:
    if p.exists():
        CKPT_PATH = p
        break

if CKPT_PATH is None:
    raise FileNotFoundError(
        "No GenD WildDeepfake checkpoint found. Checked:\n" +
        "\n".join(str(p) for p in CKPT_CANDIDATES)
    )

# In-domain target: WildDeepfake canonical test split.
TEST_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical_gend_safe.csv"
if not TEST_MANIFEST.exists():
    TEST_MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

OUT_DIR = PROJECT_ROOT / "outputs_gend_wilddeepfake_indomain_test_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = "test"
IMAGE_SIZE = 224
FRAMES_PER_VIDEO = 32
BATCH_SIZE = 16
NUM_WORKERS = 0
UNCERTAINTY_CONFIDENCE_THRESHOLD = 0.60
SEED = 42

print("GenD script:", GEND_SCRIPT)
print("Checkpoint:", CKPT_PATH)
print("Test manifest:", TEST_MANIFEST)
print("Output dir:", OUT_DIR)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def normalize_manifest(path, split="test"):
    df = pd.read_csv(path)

    if "frame_path" not in df.columns:
        if "image_path" in df.columns:
            df["frame_path"] = df["image_path"].astype(str)
        else:
            raise ValueError("Manifest needs frame_path or image_path.")

    if "video_id" not in df.columns:
        if "sample_unit_id" in df.columns:
            df["video_id"] = df["sample_unit_id"].astype(str)
        elif "group_id" in df.columns:
            df["video_id"] = df["group_id"].astype(str)
        else:
            df["video_id"] = df["frame_path"].apply(lambda x: str(Path(x).parent))

    required = ["frame_path", "split", "label", "video_id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing manifest columns: {missing}")

    df["frame_path"] = df["frame_path"].astype(str)
    df["video_id"] = df["video_id"].astype(str)
    df["split"] = df["split"].astype(str)
    df["label"] = df["label"].astype(int)

    df = df[df["split"] == split].copy()

    if len(df) == 0:
        raise RuntimeError(f"No rows found for split={split}")

    return df


def sample_frames_per_video(df, frames_per_video=32):
    parts = []

    for vid, g in df.groupby("video_id", sort=False):
        g = g.sort_values("frame_path").reset_index(drop=True)

        if len(g) <= frames_per_video:
            parts.append(g)
        else:
            idx = np.linspace(0, len(g) - 1, frames_per_video).round().astype(int)
            parts.append(g.iloc[idx].copy())

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out


class ManifestImageDataset(Dataset):
    def __init__(self, df, image_size=224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["frame_path"]
        label = int(row["label"])
        video_id = str(row["video_id"])

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.image_size, self.image_size))

        return {
            "image": self.tf(img),
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
            "video_id": video_id,
        }


def import_gend_module(script_path):
    spec = importlib.util.spec_from_file_location("gend_reimpl_ffpp_runtime", str(script_path))
    module = importlib.util.module_from_spec(spec)
    sys.modules["gend_reimpl_ffpp_runtime"] = module
    spec.loader.exec_module(module)
    return module


def get_state_dict_from_checkpoint(ckpt):
    if isinstance(ckpt, dict):
        for key in ["model", "model_state_dict", "state_dict", "net", "network"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                return ckpt[key]
    if isinstance(ckpt, dict):
        return ckpt
    raise RuntimeError(f"Unsupported checkpoint type: {type(ckpt)}")


def clean_state_dict_keys(state):
    cleaned = {}
    for k, v in state.items():
        nk = str(k)
        for prefix in ["module.", "model.", "net.", "network."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned


def build_gend_model(gend_module, ckpt):
    if not hasattr(gend_module, "GenDReimpl"):
        raise AttributeError("Could not find GenDReimpl in gend_reimpl_ffpp.py")

    ModelCls = gend_module.GenDReimpl

    cfg = {}
    if isinstance(ckpt, dict):
        for key in ["config", "args", "hparams", "hyperparameters"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                cfg.update(ckpt[key])

    sig = inspect.signature(ModelCls)
    kwargs = {}

    # Use checkpoint config when available, otherwise use common script defaults.
    for name in sig.parameters:
        if name == "self":
            continue
        if name in cfg:
            kwargs[name] = cfg[name]
        elif name == "backbone":
            kwargs[name] = cfg.get("backbone", "resnet18")
        elif name == "pretrained":
            kwargs[name] = False
        elif name == "num_classes":
            kwargs[name] = 2
        elif name == "train_ln":
            kwargs[name] = cfg.get("train_ln", False)
        elif name == "dropout":
            kwargs[name] = cfg.get("dropout", 0.0)

    print("GenD constructor kwargs:", kwargs)

    try:
        model = ModelCls(**kwargs)
    except Exception as e:
        print("Constructor with kwargs failed:", repr(e))
        print("Trying GenDReimpl() with defaults.")
        model = ModelCls()

    return model


def extract_logits(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, dict):
        for key in [
            "logits", "cls", "pred", "prediction", "output", "out",
            "y_pred", "final_output", "prob"
        ]:
            if key in output and torch.is_tensor(output[key]):
                return output[key]

        for value in output.values():
            if torch.is_tensor(value) and value.ndim >= 1:
                return value

    if isinstance(output, (list, tuple)):
        for value in output:
            if torch.is_tensor(value):
                return value

        for value in output:
            try:
                return extract_logits(value)
            except Exception:
                pass

    raise RuntimeError(f"Could not extract logits from output type: {type(output)}")


def forward_gend(model, images):
    attempts = [
        lambda: model(images),
        lambda: model({"image": images}),
        lambda: model({"img": images}),
        lambda: model({"x": images}),
    ]

    errors = []
    for fn in attempts:
        try:
            output = fn()
            logits = extract_logits(output)

            if logits.ndim == 1:
                logits = logits.view(-1, 1)

            if logits.ndim > 2:
                logits = logits.view(logits.shape[0], -1)

            return logits

        except Exception as e:
            errors.append(repr(e))

    raise RuntimeError("GenD forward failed. Errors:\n" + "\n".join(errors))


def fake_scores_from_logits(logits):
    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])
    return torch.softmax(logits[:, :2], dim=1)[:, 1]


def compute_eer(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if len(np.unique(y_true)) < 2:
        return np.nan

    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0)


def compute_uncertain_rate(y_score, threshold=0.60):
    y_score = np.asarray(y_score).astype(float)
    confidence = np.maximum(y_score, 1.0 - y_score)
    return float(np.mean(confidence < threshold) * 100.0)


def compute_metric_row(y_true, y_score, level, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    return {
        "Level": level,
        "Decision Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "AUPRC": average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "AUCROC": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "EER (Equal Error Rate)": compute_eer(y_true, y_score),
        "Precision (0)": precision[0],
        "Recall (0)": recall[0],
        "F1 (0)": f1[0],
        "Precision (1)": precision[1],
        "Recall (1)": recall[1],
        "F1 (1)": f1[1],
        "Uncertain Rate (%)": compute_uncertain_rate(y_score, threshold=UNCERTAINTY_CONFIDENCE_THRESHOLD),
        "Predicted Real Count": int((y_pred == 0).sum()),
        "Predicted Fake Count": int((y_pred == 1).sum()),
        "Rows": int(len(y_true)),
    }


# ------------------------------------------------------------
# Load WildDeepfake test split
# ------------------------------------------------------------

test_df = normalize_manifest(TEST_MANIFEST, split=SPLIT)
test_df = sample_frames_per_video(test_df, frames_per_video=FRAMES_PER_VIDEO)

test_df.to_csv(OUT_DIR / "test_manifest_used.csv", index=False)

print("\nWildDeepfake test rows after sampling:", len(test_df))
print("WildDeepfake test videos:", test_df["video_id"].nunique())
print("\nTest counts:")
display(pd.crosstab(test_df["split"], test_df["label"]))

# ------------------------------------------------------------
# Load GenD model
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gend_module = import_gend_module(GEND_SCRIPT)
ckpt = torch.load(CKPT_PATH, map_location=device)

model = build_gend_model(gend_module, ckpt).to(device)

state = clean_state_dict_keys(get_state_dict_from_checkpoint(ckpt))
missing, unexpected = model.load_state_dict(state, strict=False)

print("\nLoaded GenD checkpoint.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if len(missing) > 0:
    print("First missing keys:", missing[:10])
if len(unexpected) > 0:
    print("First unexpected keys:", unexpected[:10])

model.eval()

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

loader = DataLoader(
    ManifestImageDataset(test_df, image_size=IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

all_rows = []
y_true = []
y_score = []

start_total = time.time()
num_images = 0

with torch.no_grad():
    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.time()
        logits = forward_gend(model, images)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - t0

        scores = fake_scores_from_logits(logits).detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        paths = batch["path"]
        vids = batch["video_id"]

        bs = len(labels_np)
        num_images += bs

        for i in range(bs):
            p_fake = float(scores[i])
            confidence = max(p_fake, 1.0 - p_fake)
            pred = int(p_fake >= 0.5)
            uncertain = int(confidence < UNCERTAINTY_CONFIDENCE_THRESHOLD)

            all_rows.append({
                "image_path": paths[i],
                "video_id": vids[i],
                "label": int(labels_np[i]),
                "p_fake": p_fake,
                "prediction": pred,
                "confidence": confidence,
                "uncertain": uncertain,
                "inference_time_sec_per_image_batch_avg": elapsed / max(bs, 1),
            })

        y_true.extend(labels_np.tolist())
        y_score.extend(scores.tolist())

total_elapsed = time.time() - start_total

frame_pred_df = pd.DataFrame(all_rows)
frame_pred_df.to_csv(OUT_DIR / "test_frame_predictions.csv", index=False)

# Video-level aggregation.
video_pred_df = (
    frame_pred_df
    .groupby("video_id", as_index=False)
    .agg(
        label=("label", "first"),
        p_fake=("p_fake", "mean"),
        frame_count=("p_fake", "size"),
    )
)

video_pred_df["prediction"] = (video_pred_df["p_fake"] >= 0.5).astype(int)
video_pred_df["confidence"] = np.maximum(video_pred_df["p_fake"], 1.0 - video_pred_df["p_fake"])
video_pred_df["uncertain"] = (video_pred_df["confidence"] < UNCERTAINTY_CONFIDENCE_THRESHOLD).astype(int)

video_pred_df.to_csv(OUT_DIR / "test_video_predictions.csv", index=False)

# Metrics.
frame_metrics = compute_metric_row(
    frame_pred_df["label"].values,
    frame_pred_df["p_fake"].values,
    level="Frame",
)

video_metrics = compute_metric_row(
    video_pred_df["label"].values,
    video_pred_df["p_fake"].values,
    level="Video",
)

metrics_df = pd.DataFrame([frame_metrics, video_metrics])
metrics_df.to_csv(OUT_DIR / "test_metrics.csv", index=False)

time_df = pd.DataFrame([{
    "num_images": int(num_images),
    "total_inference_time_sec": total_elapsed,
    "avg_inference_time_sec_per_image": total_elapsed / max(num_images, 1),
    "images_per_second": num_images / max(total_elapsed, 1e-9),
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "frames_per_video": FRAMES_PER_VIDEO,
}])
time_df.to_csv(OUT_DIR / "test_inference_time.csv", index=False)

print("\nSaved:")
print(OUT_DIR / "test_metrics.csv")
print(OUT_DIR / "test_frame_predictions.csv")
print(OUT_DIR / "test_video_predictions.csv")
print(OUT_DIR / "test_inference_time.csv")

print("\nGenD in-domain WildDeepfake test metrics:")
display(metrics_df)

print("\nInference time:")
display(time_df)

In [2]:
!pip install einops

In [3]:
!pip install numpy transformers torchvision scikit-learn pandas pillow einops

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)


In [6]:
from pathlib import Path
import subprocess
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

SCRIPT = PROJECT_ROOT / "DFD-HR/run_dfd_hr_df40.py"
MANIFEST = PROJECT_ROOT / "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"

TRAIN_OUT = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_train_no_amp_cc"
CKPT = TRAIN_OUT / "latest_checkpoint.pt"

OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_latest_test"
LOG_PATH = OUT_DIR / "dfdhr_latest_test.log"
PID_PATH = OUT_DIR / "dfdhr_latest_test.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

if not MANIFEST.exists():
    raise FileNotFoundError(MANIFEST)

if not CKPT.exists():
    raise FileNotFoundError(f"Missing latest checkpoint: {CKPT}")

cmd = [
    "python", "-u", str(SCRIPT),
    "--repo_root", str(PROJECT_ROOT / "DFD-HR"),
    "--metadata_csv", str(MANIFEST),
    "--ckpt_path", str(CKPT),
    "--out_dir", str(OUT_DIR),
    "--split", "test",
    "--image_size", "448",
    "--frames_per_video", "32",
    "--batch_size", "8",
    "--num_workers", "4",
    "--path_col", "frame_path",
    "--label_col", "label",
    "--split_col", "split",
    "--group_col", "video_id",
    "--allow_non_strict",
]

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Checkpoint:", CKPT)
print("Output:", OUT_DIR)
print("Log:", LOG_PATH)

bash_cmd = (
    "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
    "PYTHONUNBUFFERED=1 nohup " + " ".join(shlex.quote(x) for x in cmd) +
    " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
)

proc = subprocess.run(
    ["bash", "-lc", bash_cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

pid = proc.stdout.strip().splitlines()[-1]
PID_PATH.write_text(pid)

print("\nLaunched DFD-HR test PID:", pid)
print("PID saved to:", PID_PATH)

Command:
python -u '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR/run_dfd_hr_df40.py' --repo_root '/home/jovyan/Deepfake_Low Resource_Low Resolution/DFD-HR' --metadata_csv '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv' --ckpt_path '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/latest_checkpoint.pt' --out_dir '/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_latest_test' --split test --image_size 448 --frames_per_video 32 --batch_size 8 --num_workers 4 --path_col frame_path --label_col label --split_col split --group_col video_id --allow_non_strict
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/latest_checkpoint.pt
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_latest_test
Log: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wildde

In [7]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_dfdhr_wilddeepfake_latest_test"
LOG_PATH = OUT_DIR / "dfdhr_latest_test.log"
PID_PATH = OUT_DIR / "dfdhr_latest_test.pid"

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout or "No GPU process found.")

print("\nOutput files:")
for name in [
    "test_metrics.csv",
    "test_frame_predictions.csv",
    "test_video_predictions.csv",
    "test_threshold_report.csv",
    "test_inference_time.csv",
    "dataset_summary.csv",
    "dfdhr_latest_test.log",
]:
    p = OUT_DIR / name
    if p.exists():
        print(f"FOUND   {name:35s} size_mb={p.stat().st_size/1024**2:.3f} age_min={(time.time()-p.stat().st_mtime)/60:.1f}")
    else:
        print(f"MISSING {name}")

print("\nMetrics:")
for name in ["test_metrics.csv", "metrics.csv"]:
    p = OUT_DIR / name
    if p.exists():
        display(pd.read_csv(p))
        break

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

PID: 1939
    PID    PPID     ELAPSED STAT %CPU %MEM   RSS CMD


GPU:
902, python, 4078


Output files:
MISSING test_metrics.csv
MISSING test_frame_predictions.csv
MISSING test_video_predictions.csv
MISSING test_threshold_report.csv
MISSING test_inference_time.csv
FOUND   dataset_summary.csv                 size_mb=4.403 age_min=0.1
FOUND   dfdhr_latest_test.log               size_mb=0.006 age_min=0.0

Metrics:

Log tail:
Log lines: 94
Device: cuda
Metadata: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_train_no_amp_cc/latest_checkpoint.pt
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_dfdhr_wilddeepfake_latest_test
Dropped missing paths: 0

Sampling videos: 100%|██████████| 806/806 [00:07<00:00, 108.39it/s]
Rows: 25792
Videos/groups: 806
col_0  count
label       
0      12672
1      13120
Traceback (most recent call la

In [4]:
!pip install einops

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
